# Spike Sorting Unit Quality Prediction — Prototype

**Goal:** Train XGBoost models to predict `fmiss` and `fpos` from features computed without ground truth.

**Training data:** SpikeForest (kachery-cloud) — legacy sorters on synthetic recordings  
**Test data:** sim_hybrid_KS042 (Janelia figshare) — modern sorters, completely held-out  
**Leakage prevention:** GroupKFold on `recording_name` — same GT unit never spans train/val

---
**Runtime estimate:** ~90 min (free Colab) | **RAM required:** ~8 GB peak

## 📦 0. Install & Restart Runtime
> Run this cell, then **Runtime → Restart session**, then skip to Cell 2.

In [ ]:
import subprocess
import sys


def run_pip(packages, label, strict=False, force_reinstall=False):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        "--prefer-binary",
        *( ["--force-reinstall"] if force_reinstall else [] ),
        *packages,
    ]
    proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").splitlines()[-16:]
        print(f"⚠ {label} install failed (exit={proc.returncode})")
        if tail:
            print(*tail, sep="\n")
        if strict:
            raise RuntimeError(f"Failed to install {label}")
        return False
    print(f"✓ {label} installed")
    return True


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=False,
)

# Core numerical stack: force NumPy<2 to avoid ABI break with compiled extensions.
run_pip(
    [
        "numpy==1.26.4",
        "pandas==2.2.3",
        "scipy==1.14.1",
        "scikit-learn==1.5.2",
        "pyarrow==18.1.0",
        "numba==0.60.0",
        "xgboost==2.0.3",
    ],
    "core numerical stack (numpy<2)",
    strict=True,
    force_reinstall=True,
)

# Spike sorting stack
si_ok = run_pip(
    [
        "numcodecs==0.15.1",
        "zarr==2.18.5",
        "mtscomp",
        "spikeinterface>=0.103.0,<0.104.0",
        "spikeforest",
        "kachery-cloud",
    ],
    "SpikeInterface stack",
    strict=False,
)
if not si_ok:
    run_pip(
        [
            "numcodecs==0.15.1",
            "zarr==2.18.5",
            "mtscomp",
            "spikeinterface",
            "spikeforest",
            "kachery-cloud",
        ],
        "SpikeInterface fallback stack",
        strict=True,
    )

# Utility/plot packages
run_pip(
    [
        "matplotlib",
        "seaborn",
        "rich",
        "psutil",
        "tqdm",
        "wandb",
    ],
    "utility stack",
    strict=False,
)

# SHAP is optional. Keep non-fatal so the whole notebook doesn't fail.
shap_ok = run_pip(["shap==0.46.0"], "SHAP optional", strict=False)
if not shap_ok:
    print("⚠ SHAP optional install failed. The notebook will skip SHAP plots.")

# Hard compatibility smoke checks
numcodecs_smoke = subprocess.run(
    [sys.executable, "-c", "from numcodecs.blosc import cbuffer_sizes, cbuffer_metainfo"],
    check=False,
)
if numcodecs_smoke.returncode != 0:
    run_pip(["numcodecs==0.15.1", "zarr==2.18.5"], "numcodecs/zarr repair", strict=True, force_reinstall=True)
    numcodecs_smoke = subprocess.run(
        [sys.executable, "-c", "from numcodecs.blosc import cbuffer_sizes, cbuffer_metainfo"],
        check=False,
    )
    if numcodecs_smoke.returncode != 0:
        raise RuntimeError("numcodecs/zarr compatibility check failed.")

core_smoke = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy as np; import numpy.random.mtrand; import pandas; import scipy; import sklearn; import xgboost; assert int(np.__version__.split('.')[0]) < 2",
    ],
    check=False,
)
if core_smoke.returncode != 0:
    raise RuntimeError("Core stack smoke test failed (NumPy/Pandas/SciPy/sklearn/xgboost).")

si_smoke = subprocess.run(
    [
        sys.executable,
        "-c",
        "import spikeinterface.core as si; import spikeinterface.comparison as sc; import spikeinterface.qualitymetrics as sqm",
    ],
    check=False,
)
if si_smoke.returncode != 0:
    raise RuntimeError("SpikeInterface import smoke test failed in install cell.")

def clear_loaded_modules(prefixes):
    import gc
    import sys as _sys
    for name in list(_sys.modules):
        if any(name == pfx or name.startswith(f"{pfx}.") for pfx in prefixes):
            _sys.modules.pop(name, None)
    gc.collect()


def ensure_inprocess_core_abi():
    try:
        clear_loaded_modules(["numpy", "pandas", "scipy"])
        import numpy as _np
        import numpy.random.mtrand  # noqa: F401
        import pandas as _pd  # noqa: F401
        import scipy as _sp  # noqa: F401
        if int(str(_np.__version__).split(".")[0]) >= 2:
            raise RuntimeError(f"NumPy {_np.__version__} detected; requires numpy<2")
        return
    except Exception:
        print("⚠ In-kernel ABI mismatch detected; repairing core stack...")
        run_pip(
            [
                "numpy==1.26.4",
                "pandas==2.2.3",
                "scipy==1.14.1",
                "scikit-learn==1.5.2",
                "xgboost==2.0.3",
            ],
            "in-kernel core ABI repair",
            strict=True,
            force_reinstall=True,
        )
        clear_loaded_modules(["numpy", "pandas", "scipy", "sklearn", "xgboost"])
        import numpy as _np2
        import numpy.random.mtrand  # noqa: F401
        import pandas as _pd2  # noqa: F401
        import scipy as _sp2  # noqa: F401
        if int(str(_np2.__version__).split(".")[0]) >= 2:
            raise RuntimeError(f"NumPy {_np2.__version__} still incompatible after repair")


ensure_inprocess_core_abi()

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    check=False,
    capture_output=True,
    text=True,
)

print("✓ Install complete. Continue to next cell in the same session.")
if pip_check.returncode != 0:
    print("⚠ `pip check` reported base-image conflicts (often harmless in Colab).")
    print("  The imports cell performs strict runtime compatibility checks.")



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ 1. Configuration — Edit This Cell Only

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIGURATION  —  edit this cell only
# ════════════════════════════════════════════════════

import os
import random
from pathlib import Path

# ── Root layout: persist only curated artifacts to Drive ─────────────
PROTOTYPE_ROOT = Path("/content/drive/MyDrive/Thesis/prototype")
ARTIFACTS_DIR  = PROTOTYPE_ROOT / "artifacts"
DATA_DIR       = ARTIFACTS_DIR / "features"
MODEL_DIR      = ARTIFACTS_DIR / "models"
RESULTS_DIR    = ARTIFACTS_DIR / "results"
FIGURES_DIR    = RESULTS_DIR / "figures"
MANIFEST_DIR   = ARTIFACTS_DIR / "manifests"
AUDIT_DIR      = ARTIFACTS_DIR / "audits"
CHECKPOINT_DIR = PROTOTYPE_ROOT / "checkpoints"
LOG_DIR        = PROTOTYPE_ROOT / "logs"
KACHERY_CLOUD_DIR = PROTOTYPE_ROOT / ".kachery-cloud"

# ── External source data (referenced in-place, not duplicated) ───────
SOURCE_DATA_ROOT = Path("/content/drive/MyDrive/Thesis/Data")
SIM_HYBRID_ROOT  = SOURCE_DATA_ROOT / "Simulations_Kilosort4_Extracted"

# ── Scratch space for heavy temporary compute only ────────────────────
SCRATCH_ROOT       = Path("/content/spike_qc_cache")
DOWNLOAD_CACHE_DIR = SCRATCH_ROOT / "downloads"
SORTER_SCRATCH_DIR = SCRATCH_ROOT / "sorters"

# ── Persistent checkpoints (Drive) for crash-safe resume ─────────────
TRAIN_PARTS_DIR      = CHECKPOINT_DIR / "train_feature_parts"
TEST_PARTS_DIR       = CHECKPOINT_DIR / "test_feature_parts"
CV_CHECKPOINT_DIR    = CHECKPOINT_DIR / "cv"
PREPROCESS_DIR       = CV_CHECKPOINT_DIR / "preprocessing"
OOF_DIR              = CV_CHECKPOINT_DIR / "oof_predictions"
DIAGNOSTIC_DIR       = CHECKPOINT_DIR / "diagnostics"

for d in [
    PROTOTYPE_ROOT, ARTIFACTS_DIR, DATA_DIR, MODEL_DIR, RESULTS_DIR, FIGURES_DIR,
    KACHERY_CLOUD_DIR,
    MANIFEST_DIR, AUDIT_DIR, CHECKPOINT_DIR, LOG_DIR, SCRATCH_ROOT,
    DOWNLOAD_CACHE_DIR, SORTER_SCRATCH_DIR, TRAIN_PARTS_DIR, TEST_PARTS_DIR,
    CV_CHECKPOINT_DIR, PREPROCESS_DIR, OOF_DIR, DIAGNOSTIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET             = DATA_DIR / "features_train.parquet"
TEST_PARQUET              = DATA_DIR / "features_test.parquet"
TRAIN_FEATURE_STATE_JSON  = CHECKPOINT_DIR / "train_feature_state.json"
TEST_FEATURE_STATE_JSON   = CHECKPOINT_DIR / "test_feature_state.json"
CV_STATE_JSON             = CHECKPOINT_DIR / "cv_state.json"
DATA_AUDIT_JSON           = AUDIT_DIR / "data_audit.json"
LEAKAGE_REPORT_JSON       = AUDIT_DIR / "leakage_risks.json"
ARTIFACT_MANIFEST_JSON    = MANIFEST_DIR / "artifact_manifest.json"
FEATURE_COLUMNS_JSON      = MANIFEST_DIR / "feature_columns.json"
FULL_TRAIN_MEDIANS_JSON   = MANIFEST_DIR / "full_train_feature_medians.json"
TEST_SOURCE_JSON          = MANIFEST_DIR / "test_source_manifest.json"
OOF_PREDICTIONS_PARQUET   = RESULTS_DIR / "oof_predictions.parquet"
TEST_PREDICTIONS_PARQUET  = RESULTS_DIR / "test_predictions.parquet"
RUN_MANIFEST_JSON         = MANIFEST_DIR / "run_manifest.json"
SOURCE_DATA_MANIFEST_JSON = MANIFEST_DIR / "source_data_manifest.json"
MODEL_MANIFEST_JSON       = MANIFEST_DIR / "model_manifest.json"
FOLD_ASSIGNMENTS_PARQUET  = RESULTS_DIR / "cv_fold_assignments.parquet"

# ── Resume / rebuild policy ───────────────────────────────────────────
RESUME_FROM_CHECKPOINTS      = True
REUSE_EXISTING_TRAIN_FEATURES = True
REUSE_EXISTING_TEST_FEATURES  = True
FORCE_REBUILD_TRAIN_FEATURES  = False
FORCE_REBUILD_TEST_FEATURES   = False
FORCE_RETRAIN_MODELS          = False
CLEAN_SCRATCH_AFTER_RUN       = True
PHASE1_MAX_ITEMS              = None  # set e.g. 3 for debug batches

# ── Sequential storage strategy ───────────────────────────────────────
# Each recording/sorter output is processed independently and written to
# Drive immediately. If the runtime crashes, reruns skip completed parts
# unless one of the FORCE_REBUILD_* flags is set to True.

# ── SpikeForest training study sets ───────────────────────────────────
# Includes both paired (real) ground-truth recordings and synthetic study sets.
# Synthetic sets add more labelled units for training without extra data collection.
# SYNTH_* sets are fetched through the same SpikeForest API as PAIRED_* sets.
SPIKEFOREST_STUDY_SETS = [
    # ── Paired (real) ground-truth recordings ───────────────────────
    "PAIRED_BOYDEN",
    "PAIRED_CRCNS_HC1",
    "PAIRED_ENGLISH",
    "PAIRED_KAMPFF",
    "PAIRED_MEA64C_YGER",
    # ── Synthetic study sets (ground truth available) ────────────────
    # SYNTH_JANELIA:  60 recordings, ~50 units, 4–64 ch, NP-noise, drift variants
    "SYNTH_JANELIA",
    # SYNTH_BIONET:   36 recordings, ~40 units, 60 ch, biophysically realistic
    "SYNTH_BIONET",
    # SYNTH_MEAREC_NEURONEX: 60 recordings, ~10 units, 32 ch, MEArec, varying SNR
    "SYNTH_MEAREC_NEURONEX",
    # SYNTH_MAGLAND:  80 recordings, 10 units, 4–8 ch, simple noise ("K10")
    "SYNTH_MAGLAND",
    # SYNTH_VISAPY:   6 recordings, ~10 units, 30 ch; KS/KS2 crash on these —
    #   per-key failures are already caught and logged to failed_keys, so safe.
    "SYNTH_VISAPY",
    # ── Low-yield synthetics (optional; comment out to save time) ────
    # SYNTH_MEAREC_TETRODE: 40 recordings, 4-ch tetrode — limited channel diversity
    # "SYNTH_MEAREC_TETRODE",
    # SYNTH_MONOTRODE: 1 recording, 1 channel — trivially small, skip by default
    # "SYNTH_MONOTRODE",
    # ── Hybrid (real recording + injected spikes) — needs SPIKEFOREST_EXTRA_URI_PAIRS
    # ⚠ leakage guard below excludes any recording matching TEST_RECORDING_SPECS
    "HYBRID_JANELIA",
]

# Optional URI overrides (None = SpikeForest package defaults)
SPIKEFOREST_SORTING_OUTPUTS_URI = None
SPIKEFOREST_RECORDINGS_URI = None

# Additional sorting-output + recording URIs merged on top of the default.
# HYBRID_JANELIA is available but lives in a separate kachery URI — add it here.
# Each entry is (sorting_outputs_uri, recordings_uri); set recordings_uri=None
# if the default recordings JSON already covers it.
SPIKEFOREST_EXTRA_URI_PAIRS = [
    (
        # HYBRID_JANELIA sorting outputs
        "sha1://9259d3ec1d981560e35c2ca41e59c39f2af3d37e?label=spikeforest-sorting-outputs.json",
        # HYBRID_JANELIA recordings (needed for recording extractor + GT)
        "sha1://43298d72b2d0860ae45fc9b0864137a976cb76e8?hybrid-janelia-spikeforest-recordings.json",
    ),
]

# ── Leakage guard: training recordings to exclude ───────────────────
# Any recording_name that appears in the test set is excluded from training
# to prevent the same underlying session appearing in both splits.
# Populated automatically from TEST_RECORDING_SPECS below.
TRAIN_EXCLUDE_RECORDING_NAMES: set = set()   # filled after TEST_RECORDING_SPECS is defined

# ── Test set: use existing Drive assets first ────────────────────────
TEST_SORTERS = ["kilosort4"]
RUN_TEST_SORTERS = False
TEST_RECORDING_SPECS = [
    {
        "recording_name": "sim_hybrid_KS042_2020-11-23",
        "recording_dir": str(SIM_HYBRID_ROOT / "sim_hybrid_KS042_2020-11-23"),
        "ground_truth_candidates": [
            str(SIM_HYBRID_ROOT / "sim_hybrid_KS042_2020-11-23" / "sim.imec0.ap_params.npz"),
            str(SIM_HYBRID_ROOT / "sim_hybrid_KS042_2020-11-23" / "sim.imec0.ap_params"),
        ],
        "precomputed_sorters": {
            "kilosort4": str(
                SIM_HYBRID_ROOT
                / "sim_hybrid_KS042_2020-11-23"
                / "kilosort4"
            ),
        },
    },
]

# ── Feature extraction ────────────────────────────────────────────────
MAX_SPIKES_TEMPLATE  = 500
MAX_SPIKES_ACG       = 10000
MAX_SPIKES_AMP       = 2000
ACG_MAX_LAG_MS       = 50
ACG_BIN_SIZE_MS      = 2.5
COSINE_THRESHOLD     = 0.7
AMPLITUDE_THRESHOLD  = 0.25
TEMPLATE_WINDOW_MS   = 1.5
AMP_WINDOW_SAMPLES   = 15

# ── Validation protocol ───────────────────────────────────────────────
TRAIN_GROUP_COLS           = ["study_set", "study_name", "recording_name"]
INNER_EVAL_GROUP_TEST_SIZE = 0.2
EARLY_STOPPING_ROUNDS      = 50

# ── Model hyperparameters ─────────────────────────────────────────────
XGB_PARAMS = dict(
    tree_method="hist",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mae",
    n_jobs=1,
    random_state=42,
)
CV_FOLDS  = 5
RAND_SEED = 42
DETERMINISTIC_NUM_THREADS = 1

# ── Reproducibility ──────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = str(RAND_SEED)
os.environ["KACHERY_CLOUD_DIR"] = str(KACHERY_CLOUD_DIR)
for env_name in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
]:
    os.environ[env_name] = str(DETERMINISTIC_NUM_THREADS)
random.seed(RAND_SEED)
# NumPy seeding is done in the imports cell after runtime compatibility checks.

# ── W&B (optional) ───────────────────────────────────────────────────
USE_WANDB     = False
WANDB_PROJECT = "spike-qc"

print("✓ Config loaded")
print(f"  Prototype root: {PROTOTYPE_ROOT}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  External data root: {SOURCE_DATA_ROOT}")
print(f"  Scratch root: {SCRATCH_ROOT}")
print(f"  Kachery cloud dir: {KACHERY_CLOUD_DIR}")
TRAIN_EXCLUDE_RECORDING_NAMES = {r["recording_name"] for r in TEST_RECORDING_SPECS}
print(f"  Training study sets: {SPIKEFOREST_STUDY_SETS}")
print(f"  Leakage-excluded recordings: {TRAIN_EXCLUDE_RECORDING_NAMES}")
print(f"  Sorting outputs URI override: {SPIKEFOREST_SORTING_OUTPUTS_URI}")
print(f"  Recordings URI override: {SPIKEFOREST_RECORDINGS_URI}")
print(f"  Test sorters: {TEST_SORTERS}")
print(f"  Run missing test sorters: {RUN_TEST_SORTERS}")
print(f"  Resume from checkpoints: {RESUME_FROM_CHECKPOINTS}")
print(f"  Phase 1 max items: {PHASE1_MAX_ITEMS}")
print(f"  Deterministic threads: {DETERMINISTIC_NUM_THREADS}")



## 📚 2. Imports & Utilities

In [ ]:
# ── Standard ───────────────────────────────────────
import gc
import hashlib
import importlib.metadata as importlib_metadata
try:
    from packaging.version import Version, InvalidVersion
except Exception:
    Version = None
    InvalidVersion = Exception
import json
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

def clear_loaded_modules(prefixes):
    import gc
    for name in list(sys.modules):
        if any(name == pfx or name.startswith(f"{pfx}.") for pfx in prefixes):
            sys.modules.pop(name, None)
    gc.collect()


def ensure_numpy_runtime_compat():
    def _try_import_core():
        try:
            clear_loaded_modules(["numpy", "pandas", "scipy"])
            import numpy as _np
            import numpy.random.mtrand  # noqa: F401
            import pandas as _pd  # noqa: F401
            import scipy as _sp  # noqa: F401
            return _np, _pd, _sp, None
        except Exception as exc:
            return None, None, None, exc

    _np, _pd, _sp, err = _try_import_core()
    if err is not None:
        print("⚠ Repairing NumPy/Pandas/SciPy ABI in current kernel...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--force-reinstall",
                "--no-cache-dir",
                "--prefer-binary",
                "numpy==1.26.4",
                "pandas==2.2.3",
                "scipy==1.14.1",
                "scikit-learn==1.5.2",
                "xgboost==2.0.3",
            ],
            check=False,
        )
        _np, _pd, _sp, err = _try_import_core()

    if err is not None:
        raise RuntimeError(
            "NumPy binary compatibility failed in this kernel even after repair. "
            "Do one Colab factory reset runtime, then run install cell first."
        ) from err

    ver = str(_np.__version__)
    if Version is not None:
        try:
            if Version(ver) >= Version("2.0.0"):
                raise RuntimeError(
                    f"NumPy {ver} detected, but this notebook requires numpy<2 due to compiled dependency compatibility."
                )
        except InvalidVersion:
            pass
    else:
        if ver.split(".")[0].isdigit() and int(ver.split(".")[0]) >= 2:
            raise RuntimeError(
                f"NumPy {ver} detected, but this notebook requires numpy<2 due to compiled dependency compatibility."
            )

    if "RAND_SEED" in globals():
        _np.random.seed(RAND_SEED)


ensure_numpy_runtime_compat()

import numpy as np
import pandas as pd
import scipy.stats as stats

# ── Compatibility preflight ─────────────────────────
def ensure_numcodecs_blosc_compat():
    smoke = subprocess.run(
        [
            sys.executable,
            "-c",
            "from numcodecs.blosc import cbuffer_sizes, cbuffer_metainfo",
        ],
        check=False,
    )
    if smoke.returncode == 0:
        return
    print("⚠ Repairing numcodecs/zarr compatibility for SpikeInterface...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "--no-cache-dir",
            "numcodecs==0.15.1",
            "zarr==2.18.5",
        ],
        check=True,
    )
    smoke2 = subprocess.run(
        [
            sys.executable,
            "-c",
            "from numcodecs.blosc import cbuffer_sizes, cbuffer_metainfo",
        ],
        check=False,
    )
    if smoke2.returncode != 0:
        raise RuntimeError(
            "Failed to repair numcodecs/zarr compatibility. Re-run the install cell in this session."
        )

ensure_numcodecs_blosc_compat()

# ── SpikeInterface ─────────────────────────────────
def ensure_spikeinterface_available():
    def _try_import():
        try:
            import spikeinterface.core as _si
            import spikeinterface.comparison as _sc
            import spikeinterface.qualitymetrics as _sqm
            return _si, _sc, _sqm
        except Exception:
            return None

    imported = _try_import()
    if imported is not None:
        return imported

    def _run_pip_install(packages, label, strict=False):
        print(f"⚠ Installing {label}...")
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-cache-dir",
            "--prefer-binary",
            *packages,
        ]
        proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "").splitlines()[-10:]
            print(f"⚠ {label} install failed (exit={proc.returncode})")
            if tail:
                print("\n".join(tail))
            if strict:
                raise RuntimeError(f"Failed to install {label}")
            return False
        return True

    # Keep fallback minimal: avoid reinstalling already-imported core scientific libs.
    _run_pip_install(
        ["numcodecs==0.15.1", "zarr==2.18.5", "mtscomp", "spikeinterface>=0.103.0,<0.104.0"],
        "minimal SpikeInterface stack",
        strict=False,
    )
    ensure_numcodecs_blosc_compat()
    imported = _try_import()
    if imported is not None:
        return imported

    _run_pip_install(
        ["numcodecs==0.15.1", "zarr==2.18.5", "mtscomp", "spikeinterface"],
        "fallback SpikeInterface stack",
        strict=False,
    )
    ensure_numcodecs_blosc_compat()
    imported = _try_import()
    if imported is not None:
        return imported

    raise RuntimeError(
        "SpikeInterface could not be imported after repair attempts. "
        "Re-run the install cell, then re-run this imports cell in the same session."
    )

si, sc, sqm = ensure_spikeinterface_available()
try:
    import spikeforest as sf
    SPIKEFOREST_AVAILABLE = True
    SPIKEFOREST_IMPORT_ERROR = None
except Exception as exc:
    SPIKEFOREST_AVAILABLE = False
    SPIKEFOREST_IMPORT_ERROR = exc

# ── Models ─────────────────────────────────────────
import xgboost as xgb
try:
    import shap
    SHAP_AVAILABLE = True
    SHAP_IMPORT_ERROR = None
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    SHAP_IMPORT_ERROR = exc
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

# ── Monitoring ─────────────────────────────────────
import matplotlib.pyplot as plt
import psutil
import seaborn as sns
from IPython.display import display
from rich import print as rprint
from rich.console import Console
from rich.progress import (
    BarColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
    TimeRemainingColumn,
)
from rich.table import Table

warnings.filterwarnings("ignore")
console = Console()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT)

CRITICAL_VERSIONS = {
    # exact pins required for known binary compatibility
    "numcodecs": "0.15.1",
    "zarr": "2.18.5",
    # the rest are checked by compatible ranges
    "numpy": None,
    "pandas": None,
    "scipy": None,
    "scikit-learn": None,
    "pyarrow": None,
    "numba": None,
    "xgboost": None,
    "spikeinterface": None,
    "shap": None,
}

MIN_VERSION_REQUIREMENTS = {
    "numpy": "1.26.0",
    "pandas": "2.2.0",
    "scipy": "1.10.0",
    "scikit-learn": "1.5.0",
    "pyarrow": "14.0.0",
    "numba": "0.59.0",
    "xgboost": "2.0.0",
    "spikeinterface": "0.103.0",
    "shap": "0.46.0",
}

MAX_EXCLUSIVE_REQUIREMENTS = {
    # hard guard against NumPy 2 ABI break with compiled deps in this notebook
    "numpy": "2.0.0",
}

PREFERRED_VERSIONS = {
    "scikit-learn": "1.5.2",
    "xgboost": "2.0.3",
    "spikeinterface": "0.103.2",
    "shap": "0.46.0",
}

OPTIONAL_PACKAGES = {"shap"}


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "missing"


def version_at_least(installed, minimum):
    if installed == "missing":
        return False
    if Version is None:
        def _norm(v):
            parts = []
            for token in str(v).split("."):
                num = ""
                for ch in token:
                    if ch.isdigit():
                        num += ch
                    else:
                        break
                parts.append(int(num) if num else 0)
            return tuple(parts)
        return _norm(installed) >= _norm(minimum)
    try:
        return Version(installed) >= Version(minimum)
    except InvalidVersion:
        return False


def version_less_than(installed, upper):
    if installed == "missing":
        return False
    if Version is None:
        def _norm(v):
            parts = []
            for token in str(v).split("."):
                num = ""
                for ch in token:
                    if ch.isdigit():
                        num += ch
                    else:
                        break
                parts.append(int(num) if num else 0)
            return tuple(parts)
        return _norm(installed) < _norm(upper)
    try:
        return Version(installed) < Version(upper)
    except InvalidVersion:
        return False


def validate_runtime():
    table = Table(title="Runtime package audit", style="cyan")
    table.add_column("Package")
    table.add_column("Installed")
    table.add_column("Requirement")
    table.add_column("Status")

    hard_failures = []
    soft_warnings = []
    audit_packages = sorted(set(CRITICAL_VERSIONS) | set(MIN_VERSION_REQUIREMENTS) | set(MAX_EXCLUSIVE_REQUIREMENTS))

    for pkg in audit_packages:
        installed = package_version(pkg)
        exact = CRITICAL_VERSIONS.get(pkg)
        minimum = MIN_VERSION_REQUIREMENTS.get(pkg)
        upper = MAX_EXCLUSIVE_REQUIREMENTS.get(pkg)

        ok = installed != "missing"
        requirement_parts = []

        if exact is not None:
            requirement_parts.append(f"exact {exact}")
            ok = ok and installed.startswith(exact)
        if minimum is not None:
            requirement_parts.append(f">= {minimum}")
            ok = ok and version_at_least(installed, minimum)
        if upper is not None:
            requirement_parts.append(f"< {upper}")
            ok = ok and version_less_than(installed, upper)
        if not requirement_parts:
            requirement_parts.append("importable")

        requirement = ", ".join(requirement_parts)
        status = "OK" if ok else "MISMATCH"
        table.add_row(pkg, installed, requirement, status)

        is_optional = pkg in OPTIONAL_PACKAGES
        if not ok:
            if is_optional:
                soft_warnings.append((pkg, installed, f"optional; requires {requirement}"))
            else:
                hard_failures.append((pkg, installed, requirement))
        else:
            preferred = PREFERRED_VERSIONS.get(pkg)
            if preferred and installed != "missing" and not installed.startswith(preferred):
                soft_warnings.append((pkg, installed, f"preferred {preferred}"))

    table.add_row(
        "spikeforest",
        package_version("spikeforest"),
        "importable",
        "OK" if SPIKEFOREST_AVAILABLE else "UNAVAILABLE",
    )
    table.add_row(
        "shap_import",
        "OK" if SHAP_AVAILABLE else "FAILED",
        "optional",
        "OK" if SHAP_AVAILABLE else "WARN",
    )
    console.print(table)

    if soft_warnings:
        joined = ", ".join(f"{pkg}={installed} ({msg})" for pkg, installed, msg in soft_warnings)
        rprint(f"[yellow]⚠ Compatibility warnings:[/yellow] {joined}")

    if hard_failures:
        joined = ", ".join(f"{pkg}={installed} (requires {req})" for pkg, installed, req in hard_failures)
        raise RuntimeError(
            "Runtime packages are incompatible with notebook requirements. "
            f"Re-run the install cell in this session: {joined}"
        )

    if not SPIKEFOREST_AVAILABLE:
        rprint(f"[yellow]⚠ spikeforest import failed:[/yellow] {SPIKEFOREST_IMPORT_ERROR}")
    if not SHAP_AVAILABLE:
        rprint(f"[yellow]⚠ SHAP unavailable (plots will be skipped):[/yellow] {SHAP_IMPORT_ERROR}")


def bootstrap_kachery_client_keys():
    """Best-effort local key bootstrap for SpikeForest/kachery-cloud."""
    try:
        import kachery_cloud._client_keys as ck
        ck._get_client_keys_hex(generate_if_missing=True)
        pub, priv = ck._get_client_keys_hex(generate_if_missing=False)
        if pub is not None and priv is not None:
            return True
    except Exception:
        pass

    # Try kachery-cloud interactive/CLI-style init in notebook context.
    try:
        import kachery_cloud as kcl
        init_fn = getattr(kcl, "init", None)
        if callable(init_fn):
            init_fn()
        import kachery_cloud._client_keys as ck
        ck._get_client_keys_hex(generate_if_missing=True)
        pub, priv = ck._get_client_keys_hex(generate_if_missing=False)
        if pub is not None and priv is not None:
            return True
    except Exception:
        pass

    return False


def read_json(path, default=None):

    path = Path(path)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return default


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)


def reset_path(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()


def safe_stem(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "__", str(text)).strip("_")


def recording_group_key(study_set, study_name, recording_name):
    return "::".join(map(str, [study_set, study_name, recording_name]))


def make_group_ids(df):
    return (
        df[TRAIN_GROUP_COLS]
        .astype(str)
        .agg("::".join, axis=1)
        .values
    )


def ram_gb():
    vm = psutil.virtual_memory()
    return vm.used / 1e9, vm.total / 1e9


def check_memory(threshold_pct=85, label=""):
    used, total = ram_gb()
    pct = 100 * used / total
    if pct > threshold_pct:
        gc.collect()
        used2, _ = ram_gb()
        rprint(f"[yellow]⚠ RAM {pct:.0f}% at {label} — freed {used - used2:.1f}GB[/yellow]")
    return pct


def log_status(msg):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{ts}] {msg}\n"
    with open(RESULTS_DIR / "STATUS.md", "a") as f:
        f.write(entry)
    rprint(f"[green]✓[/green] {msg}")


def directory_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def write_artifact_manifest():
    manifest = []
    for f in sorted(PROTOTYPE_ROOT.rglob("*")):
        if f.is_file():
            manifest.append(
                {
                    "path": str(f),
                    "size_mb": round(f.stat().st_size / 1e6, 4),
                }
            )
    write_json(ARTIFACT_MANIFEST_JSON, manifest)
    return manifest


def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def collect_path_metadata(path, compute_hash=False, max_hash_size_mb=256):
    path = Path(path)
    meta = {
        "path": str(path),
        "exists": path.exists(),
    }
    if not path.exists():
        return meta
    stat = path.stat()
    meta.update(
        {
            "is_file": path.is_file(),
            "size_bytes": int(stat.st_size),
            "mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        }
    )
    if compute_hash and path.is_file() and stat.st_size <= max_hash_size_mb * 1_000_000:
        meta["sha256"] = sha256_file(path)
    return meta


def collect_directory_snapshot(path, max_files=50):
    path = Path(path)
    snapshot = {
        "path": str(path),
        "exists": path.exists(),
    }
    if not path.exists():
        return snapshot
    files = sorted([p for p in path.rglob("*") if p.is_file()])
    snapshot["n_files"] = len(files)
    snapshot["size_bytes"] = int(sum(p.stat().st_size for p in files))
    snapshot["sample_files"] = [
        {
            "relative_path": str(p.relative_to(path)),
            "size_bytes": int(p.stat().st_size),
        }
        for p in files[:max_files]
    ]
    return snapshot


def stable_sort_frame(df, sort_cols):
    present_cols = [c for c in sort_cols if c in df.columns]
    if not present_cols:
        return df.reset_index(drop=True)
    return df.sort_values(present_cols, kind="mergesort").reset_index(drop=True)


def make_row_uid(study_set, study_name, recording_name, sorter_name, unit_id):
    return "::".join(map(str, [study_set, study_name, recording_name, sorter_name, unit_id]))


def build_run_manifest():
    return {
        "generated_at": datetime.now().isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "rand_seed": RAND_SEED,
        "deterministic_num_threads": DETERMINISTIC_NUM_THREADS,
        "env": {
            k: os.environ.get(k)
            for k in [
                "PYTHONHASHSEED",
                "OMP_NUM_THREADS",
                "OPENBLAS_NUM_THREADS",
                "MKL_NUM_THREADS",
                "NUMEXPR_NUM_THREADS",
                "VECLIB_MAXIMUM_THREADS",
            ]
        },
        "packages": {pkg: package_version(pkg) for pkg in sorted(CRITICAL_VERSIONS)},
        "config": {
            "study_sets": SPIKEFOREST_STUDY_SETS,
            "test_sorters": TEST_SORTERS,
            "cv_folds": CV_FOLDS,
            "train_group_cols": TRAIN_GROUP_COLS,
            "xgb_params": XGB_PARAMS,
        },
    }


def build_source_data_manifest():
    records = []
    for spec in TEST_RECORDING_SPECS:
        record = {
            "recording_name": spec["recording_name"],
            "recording_dir": collect_directory_snapshot(spec["recording_dir"]),
            "ground_truth_candidates": [
                collect_path_metadata(candidate, compute_hash=True)
                for candidate in spec["ground_truth_candidates"]
            ],
            "precomputed_sorters": {
                sorter_name: collect_directory_snapshot(sorter_path)
                for sorter_name, sorter_path in spec.get("precomputed_sorters", {}).items()
            },
        }
        records.append(record)
    return {
        "generated_at": datetime.now().isoformat(),
        "source_data_root": str(SOURCE_DATA_ROOT),
        "records": records,
    }


def write_model_manifest():
    payload = {
        "generated_at": datetime.now().isoformat(),
        "feature_columns": collect_path_metadata(FEATURE_COLUMNS_JSON, compute_hash=True),
        "full_train_medians": collect_path_metadata(FULL_TRAIN_MEDIANS_JSON, compute_hash=True),
        "cv_state": collect_path_metadata(CV_STATE_JSON, compute_hash=True),
        "models": [collect_path_metadata(path, compute_hash=True) for path in sorted(MODEL_DIR.glob("model_*.json"))],
        "preprocessing": [collect_path_metadata(path, compute_hash=True) for path in sorted(PREPROCESS_DIR.glob("fold*.json"))],
        "oof_files": [collect_path_metadata(path, compute_hash=True) for path in sorted(OOF_DIR.glob("*.npz"))],
    }
    write_json(MODEL_MANIFEST_JSON, payload)
    return payload


def resolve_existing_path(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"No valid path found among: {candidates}")


def make_rng(*parts):
    key = "::".join(map(str, parts)).encode("utf-8")
    seed = int(hashlib.md5(key).hexdigest()[:8], 16) ^ RAND_SEED
    return np.random.default_rng(seed)


def sample_spike_indices(n_spikes, max_spikes, *seed_parts):
    if n_spikes <= 0:
        return np.array([], dtype=np.int64)
    rng = make_rng(*seed_parts)
    idx = rng.choice(n_spikes, min(max_spikes, n_spikes), replace=False)
    idx.sort()
    return idx.astype(np.int64)


def safe_channel_locations(recording_extractor):
    try:
        return recording_extractor.get_channel_locations()
    except Exception:
        return None


def read_spikeglx_recording(rec_dir):
    rec_dir = Path(rec_dir)
    if not rec_dir.exists():
        raise FileNotFoundError(f"Recording folder not found: {rec_dir}")
    has_meta = any(rec_dir.glob("*.ap.meta"))
    has_wave = any(rec_dir.glob("*.ap.bin")) or any(rec_dir.glob("*.ap.cbin"))
    if not has_meta or not has_wave:
        raise FileNotFoundError(
            f"{rec_dir} must contain a SpikeGLX .ap.meta file and either .ap.bin or .ap.cbin."
        )
    try:
        return si.read_spikeglx(str(rec_dir), stream_name="imec0.ap")
    except Exception:
        return si.read_spikeglx(str(rec_dir))



def load_sorting_output(sorter_name, source_path):
    source_path = Path(source_path)
    candidate_paths = [source_path, source_path / "sorter_output", source_path / "sorting"]
    last_exc = None
    for candidate in candidate_paths:
        if not candidate.exists():
            continue
        try:
            srt = si.load_extractor(str(candidate))
            if srt is not None and hasattr(srt, "get_unit_ids"):
                return srt
        except Exception as exc:
            last_exc = exc
        if sorter_name.lower().startswith("kilosort"):
            try:
                from spikeinterface.extractors.phykilosortextractors import read_kilosort
                srt = read_kilosort(str(candidate))
                if srt is not None and hasattr(srt, "get_unit_ids"):
                    return srt
            except Exception as exc:
                last_exc = exc
    raise RuntimeError(f"Could not load sorting output from {source_path}: {last_exc}")

def load_ground_truth_arrays(gt_source):
    gt_source = Path(gt_source)
    if gt_source.is_file() and gt_source.suffix == ".npz":
        with np.load(gt_source, allow_pickle=True) as data:
            arrays = {k: data[k] for k in data.files}
    elif gt_source.is_dir():
        arrays = {
            npy.stem: np.load(npy, allow_pickle=True)
            for npy in sorted(gt_source.glob("*.npy"))
        }
    else:
        raise FileNotFoundError(f"Ground-truth source not found: {gt_source}")

    spike_times = None
    labels = None
    for key in ["st", "spike_times", "samples", "arr_0"]:
        if key in arrays:
            spike_times = np.asarray(arrays[key]).astype(np.int64).ravel()
            break
    for key in ["cl", "labels", "unit_ids", "arr_1"]:
        if key in arrays:
            labels = np.asarray(arrays[key]).astype(np.int64).ravel()
            break
    if spike_times is None or labels is None:
        raise KeyError(
            f"Could not find spike-time / label arrays in {gt_source}. Keys found: {sorted(arrays.keys())}"
        )
    return spike_times, labels


def load_ground_truth_sorting(gt_source, sampling_frequency):
    spike_times, labels = load_ground_truth_arrays(gt_source)
    return si.NumpySorting.from_times_labels(
        times_list=[spike_times],
        labels_list=[labels],
        sampling_frequency=sampling_frequency,
    )


def sf_output_key(study_set, study_name, recording_name, sorter_name):
    return "/".join(map(str, [study_set, study_name, recording_name, sorter_name]))


def _sf_pick_value(obj, candidates):
    # 1) direct dict-style lookup
    if isinstance(obj, dict):
        for name in candidates:
            if name in obj and obj[name] is not None:
                return obj[name]

    # 2) attribute lookup (attribute or zero-arg method)
    for name in candidates:
        if hasattr(obj, name):
            val = getattr(obj, name)
            if callable(val):
                try:
                    val = val()
                except Exception:
                    continue
            if val is not None:
                return val

    # 3) optional to_dict()/dict conversion
    for meth in ["to_dict", "dict"]:
        if hasattr(obj, meth):
            fn = getattr(obj, meth)
            if callable(fn):
                try:
                    d = fn()
                    if isinstance(d, dict):
                        for name in candidates:
                            if name in d and d[name] is not None:
                                return d[name]
                except Exception:
                    pass

    return None


def normalize_sf_sorting_output(sf_obj):
    study_set = _sf_pick_value(sf_obj, ["study_set_name", "study_set", "studySetName"])
    study_name = _sf_pick_value(sf_obj, ["study_name", "study", "studyName"])
    recording_name = _sf_pick_value(sf_obj, ["recording_name", "recording", "recordingName"])
    sorter_name = _sf_pick_value(sf_obj, ["sorter_name", "sorter", "sorterName"])

    missing = [
        n
        for n, v in [
            ("study_set", study_set),
            ("study_name", study_name),
            ("recording_name", recording_name),
            ("sorter_name", sorter_name),
        ]
        if v is None
    ]
    if missing:
        raise RuntimeError(
            f"SpikeForest output metadata missing {missing}. "
            f"Available attrs: {[a for a in dir(sf_obj) if not a.startswith('_')][:30]}"
        )

    return {
        "study_set": str(study_set),
        "study_name": str(study_name),
        "recording_name": str(recording_name),
        "sorter_name": str(sorter_name),
    }



def load_sf_sorting_extractor(sf_obj):
    # Short-circuit: if the sorter timed out or returned a non-zero exit code, there
    # is no valid sorting output — raise early with a clear, actionable message.
    try:
        if getattr(sf_obj, "timed_out", False):
            raise RuntimeError(
                f"Sorter timed out for "
                f"{getattr(sf_obj, 'recording_name', '?')}/"
                f"{getattr(sf_obj, 'sorter_name', '?')} — no sorting output available."
            )
        rc = getattr(sf_obj, "return_code", None)
        if rc is not None:
            try:
                if int(rc) != 0:
                    raise RuntimeError(
                        f"Sorter exited with non-zero return code {rc} for "
                        f"{getattr(sf_obj, 'recording_name', '?')}/"
                        f"{getattr(sf_obj, 'sorter_name', '?')} — no sorting output available."
                    )
            except (TypeError, ValueError):
                pass
    except RuntimeError:
        raise
    except Exception:
        pass

    method_names = [
        "get_sorting_extractor",
        "sorting_extractor",
        "get_sorting",
        "load_sorting_extractor",
    ]
    for meth in method_names:
        if hasattr(sf_obj, meth):
            ref = getattr(sf_obj, meth)
            try:
                out = ref() if callable(ref) else ref
            except Exception:
                continue
            if out is not None and hasattr(out, "get_unit_ids"):
                return out
    raise RuntimeError(
        "Could not resolve a valid sorting extractor from SpikeForest output object. "
        f"Available attrs: {[a for a in dir(sf_obj) if not a.startswith('_')][:30]}"
    )

def build_templates(recording, sorting, unit_ids, fs, recording_name, sorter_name):
    templates = {}
    n_samples_total = recording.get_num_samples()
    win = int(TEMPLATE_WINDOW_MS * fs / 1000)
    for uid in unit_ids:
        st_full = sorting.get_unit_spike_train(uid, segment_index=0)
        if len(st_full) < 5:
            continue
        idx = sample_spike_indices(
            len(st_full),
            MAX_SPIKES_TEMPLATE,
            "template",
            recording_name,
            sorter_name,
            uid,
        )
        snippets = []
        for t in st_full[idx]:
            s = int(t - win)
            e = int(t + win)
            if s < 0 or e >= n_samples_total:
                continue
            snippets.append(
                recording.get_traces(start_frame=s, end_frame=e).astype(np.float32)
            )
        if snippets:
            templates[uid] = np.mean(np.stack(snippets, axis=0), axis=0).T.astype(np.float16)
    return templates



def extract_unit_amplitudes(recording, spike_times, peak_ch, recording_name, sorter_name, unit_id):
    if len(spike_times) == 0:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)
    idx = sample_spike_indices(
        len(spike_times),
        MAX_SPIKES_AMP,
        "amplitude",
        recording_name,
        sorter_name,
        unit_id,
    )
    amps = []
    amp_time_fraction = []
    n_samples_total = int(recording.get_num_samples())
    denom = float(max(n_samples_total - 1, 1))

    for t in spike_times[idx]:
        s = int(t - AMP_WINDOW_SAMPLES)
        e = int(t + AMP_WINDOW_SAMPLES)
        if s < 0 or e >= n_samples_total:
            continue
        snippet = recording.get_traces(start_frame=s, end_frame=e).astype(np.float32)
        amps.append(float(np.max(np.abs(snippet[:, peak_ch]))))
        amp_time_fraction.append(float(t) / denom)

    if len(amps) == 0:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)

    return (
        np.asarray(amps, dtype=np.float32),
        np.asarray(amp_time_fraction, dtype=np.float32),
    )


def compute_spikeinterface_unit_metrics(recording, sorting, unit_ids):
    """Compute available SpikeInterface quality metrics with robust fallbacks."""
    metric_defaults = {
        "si_num_spikes": np.nan,
        "si_firing_rate_hz": np.nan,
        "si_presence_ratio": np.nan,
        "si_isi_violations_ratio": np.nan,
        "si_isi_violations_count": np.nan,
        "si_snr": np.nan,
        "si_firing_range": np.nan,
        "si_amplitude_median": np.nan,
        "si_amplitude_cutoff": np.nan,
        "si_amplitude_cv_median": np.nan,
        "si_amplitude_cv_range": np.nan,
        "si_sd_ratio": np.nan,
        "si_drift_ptp": np.nan,
        "si_drift_std": np.nan,
        "si_drift_mad": np.nan,
    }
    out = {uid: dict(metric_defaults) for uid in unit_ids}
    if len(unit_ids) == 0:
        return out

    def _assign(uid, key, value):
        if uid not in out or key not in out[uid]:
            return
        try:
            v = float(value)
        except Exception:
            return
        if np.isfinite(v):
            out[uid][key] = v

    def _extract_value(metric_obj, uid, field=None):
        if metric_obj is None:
            return np.nan
        if isinstance(metric_obj, dict):
            return metric_obj.get(uid, np.nan)
        if isinstance(metric_obj, pd.Series):
            return metric_obj.get(uid, np.nan)
        if isinstance(metric_obj, pd.DataFrame):
            if field is not None and field in metric_obj.columns and uid in metric_obj.index:
                return metric_obj.at[uid, field]
            if uid in metric_obj.index and metric_obj.shape[1] == 1:
                return metric_obj.loc[uid].iloc[0]
            return np.nan
        if field is not None and hasattr(metric_obj, field):
            return _extract_value(getattr(metric_obj, field), uid, None)
        return np.nan

    try:
        analyzer = si.create_sorting_analyzer(
            sorting=sorting,
            recording=recording,
            format="memory",
            sparse=False,
        )
    except Exception as exc:
        print(f"⚠ SpikeInterface analyzer unavailable, using custom-only features: {exc}")
        return out

    # Fast direct metrics (generally extension-free)
    try:
        num_spikes = sqm.compute_num_spikes(analyzer, unit_ids=unit_ids)
        for uid in unit_ids:
            _assign(uid, "si_num_spikes", _extract_value(num_spikes, uid))
    except Exception:
        pass

    try:
        firing_rates = sqm.compute_firing_rates(analyzer, unit_ids=unit_ids)
        for uid in unit_ids:
            _assign(uid, "si_firing_rate_hz", _extract_value(firing_rates, uid))
    except Exception:
        pass

    try:
        presence_ratios = sqm.compute_presence_ratios(
            analyzer,
            bin_duration_s=60.0,
            mean_fr_ratio_thresh=0.0,
            unit_ids=unit_ids,
        )
        for uid in unit_ids:
            _assign(uid, "si_presence_ratio", _extract_value(presence_ratios, uid))
    except Exception:
        pass

    try:
        isi_metrics = sqm.compute_isi_violations(
            analyzer,
            isi_threshold_ms=2.0,
            min_isi_ms=0.0,
            unit_ids=unit_ids,
        )
        for uid in unit_ids:
            _assign(uid, "si_isi_violations_ratio", _extract_value(isi_metrics, uid, "isi_violations_ratio"))
            _assign(uid, "si_isi_violations_count", _extract_value(isi_metrics, uid, "isi_violations_count"))
    except Exception:
        pass

    # Compute lightweight extensions when available; ignore missing features.
    extension_jobs = [
        ("random_spikes", {"method": "uniform", "max_spikes_per_unit": 300, "seed": RAND_SEED}),
        ("waveforms", {"ms_before": 1.0, "ms_after": 2.0, "n_jobs": 1, "progress_bar": False}),
        ("templates", {}),
        ("noise_levels", {}),
        ("spike_amplitudes", {}),
        ("spike_locations", {"method": "center_of_mass"}),
    ]
    for ext_name, kwargs in extension_jobs:
        try:
            analyzer.compute(ext_name, **kwargs)
        except Exception:
            pass

    metric_names = [
        "num_spikes",
        "firing_rate",
        "presence_ratio",
        "isi_violation",
        "snr",
        "firing_range",
        "amplitude_median",
        "amplitude_cutoff",
        "amplitude_cv",
        "sd_ratio",
        "drift",
    ]
    try:
        # Filter to metrics known by this installed SpikeInterface version.
        default_qm_params = sqm.get_default_qm_params()
        available_metrics = set(default_qm_params.keys())
        metric_names = [m for m in metric_names if m in available_metrics]
    except Exception:
        pass

    qm_col_to_key = {
        "num_spikes": "si_num_spikes",
        "firing_rate": "si_firing_rate_hz",
        "presence_ratio": "si_presence_ratio",
        "isi_violations_ratio": "si_isi_violations_ratio",
        "isi_violations_count": "si_isi_violations_count",
        "snr": "si_snr",
        "firing_range": "si_firing_range",
        "amplitude_median": "si_amplitude_median",
        "amplitude_cutoff": "si_amplitude_cutoff",
        "amplitude_cv_median": "si_amplitude_cv_median",
        "amplitude_cv_range": "si_amplitude_cv_range",
        "sd_ratio": "si_sd_ratio",
        "drift_ptp": "si_drift_ptp",
        "drift_std": "si_drift_std",
        "drift_mad": "si_drift_mad",
    }

    if len(metric_names) == 0:
        return out

    try:
        qm_df = sqm.compute_quality_metrics(
            analyzer,
            metric_names=metric_names,
            skip_pc_metrics=True,
        )
        if isinstance(qm_df, pd.DataFrame) and not qm_df.empty:
            qm_df = qm_df.reindex(unit_ids)
            for uid in unit_ids:
                if uid not in qm_df.index:
                    continue
                for col, key in qm_col_to_key.items():
                    if col in qm_df.columns:
                        _assign(uid, key, qm_df.at[uid, col])
    except Exception as exc:
        print(f"⚠ SpikeInterface extended metrics partially unavailable: {exc}")

    return out


def _reindex_sorting_units_to_int(srt):
    """Return a sorting with unit IDs reindexed to int [0..n-1] and a map new->original.

    Both sortings passed to sc.compare_sorter_to_ground_truth must have identical-dtype
    unit IDs; mixing int64 and str triggers numpy ufunc errors.  This function always
    returns a sorting whose unit IDs are dense Python/numpy integers [0, 1, …, n-1].
    """
    unit_ids = list(srt.get_unit_ids())
    new_ids = list(range(len(unit_ids)))
    new_to_old = {new_id: old_id for new_id, old_id in zip(new_ids, unit_ids)}

    def _already_int_indexed(ids, expected):
        """True iff ids are already 0-based dense integers matching expected."""
        try:
            return (
                len(ids) == len(expected)
                and all(
                    isinstance(u, (int, np.integer)) and int(u) == n
                    for u, n in zip(ids, expected)
                )
            )
        except Exception:
            return False

    def _has_int_unit_ids(candidate):
        """True iff candidate.get_unit_ids() yields 0-based dense integers."""
        try:
            return _already_int_indexed(list(candidate.get_unit_ids()), new_ids)
        except Exception:
            return False

    # Fast path: already dense 0-based integer IDs (explicit type check).
    if _already_int_indexed(unit_ids, new_ids):
        return srt, new_to_old

    # Try SI-native rename/select APIs (different versions expose different signatures).
    # IMPORTANT: always verify the result actually carries integer unit IDs —
    # some SI versions accept unknown kwargs silently and ignore renamed_unit_ids.
    try:
        out = srt.select_units(unit_ids=unit_ids, renamed_unit_ids=new_ids)
        if out is not None and _has_int_unit_ids(out):
            return out, new_to_old
    except Exception:
        pass

    try:
        out = srt.select_units(unit_ids, new_ids)
        if out is not None and _has_int_unit_ids(out):
            return out, new_to_old
    except Exception:
        pass

    try:
        out = srt.rename_units(new_unit_ids=new_ids)
        if out is not None and _has_int_unit_ids(out):
            return out, new_to_old
    except Exception:
        pass

    try:
        out = srt.rename_units(new_ids)
        if out is not None and _has_int_unit_ids(out):
            return out, new_to_old
    except Exception:
        pass

    try:
        out = srt.rename_units({old: new for old, new in zip(unit_ids, new_ids)})
        if out is not None and _has_int_unit_ids(out):
            return out, new_to_old
    except Exception:
        pass

    # Last-resort fallback: rebuild as NumpySorting with canonical int labels.
    # get_unit_spike_train is called with the original IDs exactly as returned by
    # get_unit_ids() so the internal dtype lookup always matches.
    fs = float(srt.get_sampling_frequency())
    nseg = int(srt.get_num_segments())
    times_list = []
    labels_list = []
    for seg in range(nseg):
        seg_times = []
        seg_labels = []
        for new_id, old_id in zip(new_ids, unit_ids):
            try:
                st = np.asarray(
                    srt.get_unit_spike_train(old_id, segment_index=seg), dtype=np.int64
                ).ravel()
            except Exception:
                st = np.array([], dtype=np.int64)
            if st.size == 0:
                continue
            seg_times.append(st)
            seg_labels.append(np.full(st.shape, new_id, dtype=np.int64))
        if seg_times:
            tt = np.concatenate(seg_times).astype(np.int64, copy=False)
            ll = np.concatenate(seg_labels).astype(np.int64, copy=False)
            order = np.argsort(tt, kind="stable")
            times_list.append(tt[order])
            labels_list.append(ll[order])
        else:
            times_list.append(np.array([], dtype=np.int64))
            labels_list.append(np.array([], dtype=np.int64))

    rebuilt = si.NumpySorting.from_times_labels(
        times_list=times_list,
        labels_list=labels_list,
        sampling_frequency=fs,
    )
    return rebuilt, new_to_old


def _is_transient_kachery_error(exc):
    msg = str(exc).lower()
    timeout_like = [
        "timeout",
        "timed out",
        "gateway timeout",
        "function_invocation_timeout",
        "read timed out",
        "connection reset",
        "temporarily unavailable",
        "504",
    ]
    try:
        import requests as _rq
        if isinstance(exc, (_rq.exceptions.Timeout, _rq.exceptions.ConnectionError)):
            return True
    except Exception:
        pass
    return any(token in msg for token in timeout_like)


def compare_and_extract_rows(recording, sorting_gt, sorting_out, metadata):
    if sorting_out is None or not hasattr(sorting_out, "get_unit_ids"):
        raise RuntimeError("Sorting output extractor is None or invalid for this item.")
    if sorting_gt is None or not hasattr(sorting_gt, "get_unit_ids"):
        raise RuntimeError("Ground-truth sorting extractor is None or invalid for this item.")

    # Canonicalize both sortings to dense integer IDs to avoid dtype-mismatch issues
    # (int-vs-str IDs can trigger pandas/numpy comparison failures in SI pipelines).
    sorting_gt, _ = _reindex_sorting_units_to_int(sorting_gt)
    sorting_out, out_id_map = _reindex_sorting_units_to_int(sorting_out)

    # Defensive dtype guard: if reindexing still left non-integer unit IDs
    # (e.g. because all SI API variants failed silently), force the fallback
    # NumpySorting path now to guarantee int64 unit IDs on both sides before
    # sc.compare_sorter_to_ground_truth, which otherwise triggers:
    #   "ufunc 'equal' did not contain a loop with signature matching types
    #    (Int64DType, StrDType) -> None"
    def _force_numpy_int_sorting(srt_inner):
        """Unconditionally rebuild as NumpySorting with 0-based int labels."""
        ids_inner = list(srt_inner.get_unit_ids())
        fs_inner = float(srt_inner.get_sampling_frequency())
        nseg_inner = int(srt_inner.get_num_segments())
        tl, ll = [], []
        for seg in range(nseg_inner):
            st_concat, lb_concat = [], []
            for new_i, old_i in enumerate(ids_inner):
                try:
                    st = np.asarray(
                        srt_inner.get_unit_spike_train(old_i, segment_index=seg),
                        dtype=np.int64,
                    ).ravel()
                except Exception:
                    st = np.array([], dtype=np.int64)
                if st.size == 0:
                    continue
                st_concat.append(st)
                lb_concat.append(np.full(st.shape, new_i, dtype=np.int64))
            if st_concat:
                tt = np.concatenate(st_concat).astype(np.int64, copy=False)
                lb = np.concatenate(lb_concat).astype(np.int64, copy=False)
                order = np.argsort(tt, kind="stable")
                tl.append(tt[order])
                ll.append(lb[order])
            else:
                tl.append(np.array([], dtype=np.int64))
                ll.append(np.array([], dtype=np.int64))
        return si.NumpySorting.from_times_labels(
            times_list=tl, labels_list=ll, sampling_frequency=fs_inner
        )

    def _unit_ids_are_int(srt_check):
        try:
            return np.issubdtype(np.array(srt_check.get_unit_ids()).dtype, np.integer)
        except Exception:
            return False

    if not _unit_ids_are_int(sorting_gt):
        sorting_gt = _force_numpy_int_sorting(sorting_gt)
    if not _unit_ids_are_int(sorting_out):
        sorting_out = _force_numpy_int_sorting(sorting_out)
        # Rebuild out_id_map to reflect the new dense 0-based indices.
        # (The original out_id_map was built before the forced rebuild, so it
        # still maps correctly: new_int_id → original_id.)

    fs = recording.get_sampling_frequency()
    duration_sec = recording.get_num_samples() / fs
    ch_locs = safe_channel_locations(recording)
    unit_ids = list(sorting_out.get_unit_ids())
    if len(unit_ids) == 0:
        return []

    cmp = sc.compare_sorter_to_ground_truth(
        sorting_gt,
        sorting_out,
        exhaustive_gt=True,
        match_score=0.5,
        chance_score=0.1,
    )
    perf = cmp.get_performance(method="by_unit", output="dataframe")
    perf = perf.copy()
    # Keep perf index aligned with canonical sorting_out IDs.
    perf.index = perf.index.astype(np.int64, copy=False)

    block_e = extract_block_e(recording, len(unit_ids), metadata.get("study_set", ""))
    templates = build_templates(
        recording,
        sorting_out,
        unit_ids,
        fs,
        metadata["recording_name"],
        metadata["sorter_name"],
    )
    si_metrics_by_unit = compute_spikeinterface_unit_metrics(recording, sorting_out, unit_ids)

    rows = []
    for uid in unit_ids:
        if uid not in templates or uid not in perf.index:
            continue
        row_perf = perf.loc[uid]
        tmpl = templates[uid].astype(np.float32)
        st_unit = sorting_out.get_unit_spike_train(uid, segment_index=0)
        peak_ch = int(np.argmax(np.ptp(tmpl, axis=1)))
        amps, amp_time_fraction = extract_unit_amplitudes(
            recording,
            st_unit,
            peak_ch,
            metadata["recording_name"],
            metadata["sorter_name"],
            uid,
        )

        original_uid = out_id_map.get(uid, uid)
        original_uid_str = str(original_uid)

        row = dict(metadata)
        row.update(
            {
                "unit_id": original_uid_str,
                "group_key": recording_group_key(
                    metadata["study_set"],
                    metadata["study_name"],
                    metadata["recording_name"],
                ),
                "row_uid": make_row_uid(
                    metadata["study_set"],
                    metadata["study_name"],
                    metadata["recording_name"],
                    metadata["sorter_name"],
                    original_uid_str,
                ),
                "fmiss": float(1.0 - row_perf.get("recall", np.nan)),
                "fpos": float(1.0 - row_perf.get("precision", np.nan)),
                "accuracy": float(row_perf.get("accuracy", np.nan)),
            }
        )
        row.update(si_metrics_by_unit.get(uid, {}))
        row.update(extract_block_a(tmpl, ch_locs, fs))
        row.update(extract_block_b(amps, amp_time_fraction))
        row.update(extract_block_c(st_unit, fs, duration_sec))
        row.update(block_e)
        row.update(extract_block_f(uid, templates, ch_locs))
        rows.append(row)
    return rows

def ensure_feature_matrix(df, feature_cols, reference_medians):
    df = df.copy()
    missing_cols = [c for c in feature_cols if c not in df.columns]
    for col in missing_cols:
        df[col] = np.nan
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    df[feature_cols] = df[feature_cols].fillna(reference_medians)
    return df, missing_cols


def load_saved_models(target_name):
    loaded = []
    for model_path in sorted(MODEL_DIR.glob(f"model_{target_name}_fold*.json")):
        model = xgb.XGBRegressor()
        model.load_model(str(model_path))
        loaded.append(model)
    return loaded


def load_fold_medians(fold):
    payload = read_json(PREPROCESS_DIR / f"fold{fold}.json", default=None)
    if payload is None:
        raise FileNotFoundError(f"Missing preprocessing checkpoint for fold {fold}")
    return pd.Series(payload["feature_medians"], dtype=np.float32)


def load_oof_prediction(target_name, fold):
    path = OOF_DIR / f"{target_name}_fold{fold}.npz"
    if not path.exists():
        return None
    data = np.load(path)
    return {
        "val_idx": data["val_idx"].astype(np.int64),
        "preds": data["preds"].astype(np.float32),
    }


def save_oof_prediction(target_name, fold, val_idx, preds):
    np.savez_compressed(
        OOF_DIR / f"{target_name}_fold{fold}.npz",
        val_idx=np.asarray(val_idx, dtype=np.int64),
        preds=np.asarray(preds, dtype=np.float32),
    )


def upsert_metric_csv(target_name, row):
    path = RESULTS_DIR / f"cv_{target_name}.csv"
    if path.exists():
        df = pd.read_csv(path)
        df = df[df["fold"] != row["fold"]]
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df = df.sort_values("fold").reset_index(drop=True)
    df.to_csv(path, index=False)
    return df


def leakage_audit(df_train, df_val, fold):
    train_groups = set(df_train["group_key"])
    val_groups = set(df_val["group_key"])
    overlap = train_groups & val_groups
    assert len(overlap) == 0, f"LEAKAGE fold {fold}: {overlap}"
    rprint(
        f"[green]✓ Leakage audit fold {fold} passed[/green] | "
        f"train={len(train_groups)} groups | val={len(val_groups)} groups"
    )


def make_inner_group_split(groups, random_state):
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        return None
    test_groups = max(1, int(round(len(unique_groups) * INNER_EVAL_GROUP_TEST_SIZE)))
    test_size = min(len(unique_groups) - 1, test_groups) / len(unique_groups)
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_sub_idx, eval_idx = next(splitter.split(np.zeros(len(groups)), groups=groups))
    return train_sub_idx, eval_idx


validate_runtime()
write_json(RUN_MANIFEST_JSON, build_run_manifest())
write_json(SOURCE_DATA_MANIFEST_JSON, build_source_data_manifest())
print("✓ Imports complete")
used, total = ram_gb()
rprint(f"[cyan]RAM:[/cyan] {used:.1f}/{total:.1f} GB used")
rprint(f"[green]✓[/green] Run manifest saved to {RUN_MANIFEST_JSON}")
rprint(f"[green]✓[/green] Source data manifest saved to {SOURCE_DATA_MANIFEST_JSON}")




## 🔧 3. Feature Extraction Functions

In [ ]:
# ════════════════════════════════════════════════════
#  BLOCK A — Waveform Morphology
# ════════════════════════════════════════════════════

def extract_block_a(mean_template, channel_locations, fs=30000):
    """mean_template: (n_ch, n_samples) float32. Returns dict of features."""
    n_ch, n_t = mean_template.shape
    ptp_per_ch = np.ptp(mean_template, axis=1)   # (n_ch,)
    peak_ch    = np.argmax(ptp_per_ch)
    wf         = mean_template[peak_ch].astype(np.float32)
    ms_per_sample = 1000.0 / fs

    trough_idx = int(np.argmin(wf))
    peak_idx   = int(np.argmax(wf))
    ptp_uv     = float(ptp_per_ch[peak_ch])

    # Half-width: samples where wf < half of trough value
    half_val   = wf[trough_idx] / 2.0
    below      = np.where(wf < half_val)[0]
    hw_ms      = float(len(below)) * ms_per_sample if len(below) > 0 else 0.0

    # Repolarization and recovery slopes
    if trough_idx < n_t - 2:
        repol_slope   = float(wf[trough_idx+1] - wf[trough_idx]) / ms_per_sample
    else:
        repol_slope   = 0.0
    post_peak_uv  = float(np.max(wf[trough_idx:])) if trough_idx < n_t else 0.0

    # Spatial spread (in µm)
    max_ptp    = float(np.max(ptp_per_ch)) + 1e-8
    active_mask = ptp_per_ch > AMPLITUDE_THRESHOLD * max_ptp
    n_active    = int(np.sum(active_mask))

    if n_active > 0 and channel_locations is not None:
        depths     = channel_locations[active_mask, 1]   # y-axis = depth
        weights    = ptp_per_ch[active_mask]
        com_um     = float(np.average(depths, weights=weights))
        spread_um  = float(np.max(depths) - np.min(depths)) if n_active > 1 else 0.0
        spread_w   = float(np.sqrt(np.average((depths - com_um)**2, weights=weights)))
        peak_depth = float(channel_locations[peak_ch, 1])
    else:
        com_um = spread_um = spread_w = peak_depth = 0.0

    # Asymmetry
    peak_val   = float(np.max(wf))
    trough_val = float(np.min(wf))
    denom      = peak_val - abs(trough_val)
    asymmetry  = float((peak_val + abs(trough_val)) / (denom + 1e-8))

    return {
        'snr':                   ptp_uv / (float(np.std(wf)) + 1e-8),
        'peak_to_trough_uv':     ptp_uv,
        'half_width_ms':         hw_ms,
        'repolarization_slope':  repol_slope,
        'post_trough_peak_uv':   post_peak_uv,
        'trough_time_ms':        float(trough_idx) * ms_per_sample,
        'wf_energy':             float(np.sum(wf**2)),
        'wf_ptp_ratio':          ptp_uv / (float(np.std(wf)) + 1e-8),
        'wf_asymmetry':          asymmetry,
        'template_norm':         float(np.linalg.norm(wf)),
        'n_active_channels':     n_active,
        'spread_um':             spread_um,
        'center_of_mass_um':     com_um,
        'spread_weighted_um':    spread_w,
        'peak_channel_depth_um': peak_depth,
        'n_channels':            n_ch,
    }


# ════════════════════════════════════════════════════
#  BLOCK B — Amplitude Statistics
# ════════════════════════════════════════════════════


def extract_block_b(amplitudes, amp_time_fraction=None):
    """Amplitude distribution + temporal evolution features."""
    feature_names = [
        "mean",
        "std",
        "cv",
        "p5",
        "p50",
        "p95",
        "iqr",
        "skew",
        "kurtosis",
        "drift_slope",
        "drift_r2",
        "bimodality",
        "outlier_pct",
        "early_p50",
        "mid_p50",
        "late_p50",
        "early_late_ratio",
        "first_half_slope",
        "second_half_slope",
        "trend_spearman",
        "bin_cv_mean",
        "bin_cv_std",
        "bin_spread_ratio",
    ]

    if len(amplitudes) < 5:
        return {f"amp_{k}": np.nan for k in feature_names}

    amps = amplitudes[:MAX_SPIKES_AMP].astype(np.float32)
    n = len(amps)

    if amp_time_fraction is None or len(amp_time_fraction) < n:
        t_vec = np.linspace(0.0, 1.0, n, dtype=np.float32)
    else:
        t_vec = np.asarray(amp_time_fraction[:n], dtype=np.float32)
        if np.any(~np.isfinite(t_vec)):
            t_vec = np.linspace(0.0, 1.0, n, dtype=np.float32)
        else:
            t_min = float(np.min(t_vec))
            t_max = float(np.max(t_vec))
            span = max(t_max - t_min, 1e-8)
            t_vec = ((t_vec - t_min) / span).astype(np.float32)

    mu = float(np.mean(amps))
    sigma = float(np.std(amps))
    sk = float(stats.skew(amps))
    ku = float(stats.kurtosis(amps))

    if np.allclose(t_vec, t_vec[0]):
        slope = 0.0
        r2 = 0.0
    else:
        slope, _, r, _, _ = stats.linregress(t_vec, amps)
        r2 = float(r**2)

    denom_bimod = ku + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3) + 1e-8)
    bimod = (sk**2 + 1) / (denom_bimod + 1e-8)

    early_mask = t_vec <= (1.0 / 3.0)
    mid_mask = (t_vec > (1.0 / 3.0)) & (t_vec <= (2.0 / 3.0))
    late_mask = t_vec > (2.0 / 3.0)

    def _safe_median(arr):
        return float(np.median(arr)) if arr.size > 0 else np.nan

    early_p50 = _safe_median(amps[early_mask])
    mid_p50 = _safe_median(amps[mid_mask])
    late_p50 = _safe_median(amps[late_mask])
    early_late_ratio = (
        float(late_p50 / (early_p50 + 1e-8))
        if np.isfinite(early_p50) and np.isfinite(late_p50)
        else np.nan
    )

    half_mask = t_vec <= 0.5
    first_half_slope = np.nan
    second_half_slope = np.nan
    if np.sum(half_mask) >= 3 and not np.allclose(t_vec[half_mask], t_vec[half_mask][0]):
        first_half_slope = float(stats.linregress(t_vec[half_mask], amps[half_mask]).slope)
    if np.sum(~half_mask) >= 3 and not np.allclose(t_vec[~half_mask], t_vec[~half_mask][0]):
        second_half_slope = float(stats.linregress(t_vec[~half_mask], amps[~half_mask]).slope)

    trend_rho = spearmanr(t_vec, amps).statistic if n >= 6 else np.nan
    trend_rho = float(trend_rho) if trend_rho is not None and np.isfinite(trend_rho) else np.nan

    # Decile-level stability profile.
    bin_edges = np.linspace(0.0, 1.0, 11)
    bin_ids = np.digitize(t_vec, bin_edges, right=False) - 1
    bin_ids = np.clip(bin_ids, 0, 9)
    bin_cvs = []
    bin_medians = []
    for b in range(10):
        vals = amps[bin_ids == b]
        if vals.size >= 3:
            m = float(np.mean(vals))
            s = float(np.std(vals))
            bin_cvs.append(s / (abs(m) + 1e-8))
            bin_medians.append(float(np.median(vals)))

    if len(bin_cvs) == 0:
        bin_cv_mean = np.nan
        bin_cv_std = np.nan
    else:
        bin_cv_mean = float(np.mean(bin_cvs))
        bin_cv_std = float(np.std(bin_cvs))

    if len(bin_medians) < 2:
        bin_spread_ratio = np.nan
    else:
        bin_spread_ratio = float((max(bin_medians) - min(bin_medians)) / (abs(mu) + 1e-8))

    return {
        "amp_mean": mu,
        "amp_std": sigma,
        "amp_cv": sigma / (abs(mu) + 1e-8),
        "amp_p5": float(np.percentile(amps, 5)),
        "amp_p50": float(np.percentile(amps, 50)),
        "amp_p95": float(np.percentile(amps, 95)),
        "amp_iqr": float(np.percentile(amps, 75) - np.percentile(amps, 25)),
        "amp_skew": sk,
        "amp_kurtosis": ku,
        "amp_drift_slope": float(slope),
        "amp_drift_r2": r2,
        "amp_bimodality": float(bimod),
        "amp_outlier_pct": float(np.mean(np.abs(amps - mu) > 3 * sigma)),
        "amp_early_p50": early_p50,
        "amp_mid_p50": mid_p50,
        "amp_late_p50": late_p50,
        "amp_early_late_ratio": early_late_ratio,
        "amp_first_half_slope": first_half_slope,
        "amp_second_half_slope": second_half_slope,
        "amp_trend_spearman": trend_rho,
        "amp_bin_cv_mean": bin_cv_mean,
        "amp_bin_cv_std": bin_cv_std,
        "amp_bin_spread_ratio": bin_spread_ratio,
    }

def compute_acg_vectorized(spike_train_sorted, bin_size_ms, max_lag_ms, fs):
    """O(N log N) ACG. No nested loop over spike pairs."""
    max_lag_samp = int(max_lag_ms * fs / 1000)
    n_bins       = int(2 * max_lag_ms / bin_size_ms)
    acg          = np.zeros(n_bins, dtype=np.float32)
    st           = spike_train_sorted
    for t in st:
        lo   = np.searchsorted(st, t - max_lag_samp, side='left')
        hi   = np.searchsorted(st, t + max_lag_samp, side='right')
        diff = st[lo:hi] - t
        diff = diff[diff != 0]
        bins = ((diff + max_lag_samp) * n_bins / (2 * max_lag_samp)).astype(np.int32)
        v    = (bins >= 0) & (bins < n_bins)
        np.add.at(acg, bins[v], 1)
    acg /= (acg.sum() + 1e-8)
    return acg

def extract_block_c(spike_train, fs=30000, duration_sec=600):
    """spike_train: 1D int64 array of sample indices."""
    if len(spike_train) < 5:
        return {}
    st    = np.sort(spike_train.astype(np.int64))
    isis  = np.diff(st).astype(np.float32) / (fs / 1000)  # in ms
    fr    = len(st) / duration_sec

    # Presence ratio
    n_windows = int(duration_sec)
    win_samps = fs
    presence  = sum(1 for w in range(n_windows)
                    if np.searchsorted(st, (w+1)*win_samps) >
                       np.searchsorted(st, w*win_samps)) / n_windows

    # ISI violation rate
    rp_ms  = 2.0  # refractory period
    T_sec  = duration_sec
    n_viol = float(np.sum(isis < rp_ms))
    n_exp  = 2 * rp_ms * 1e-3 * len(st)**2 / (2 * T_sec)
    isi_vr = n_viol / (n_exp + 1e-8)

    # Subsampled ACG
    if len(st) > MAX_SPIKES_ACG:
        rng = np.random.default_rng(RAND_SEED)
        st_sub = np.sort(rng.choice(st, MAX_SPIKES_ACG, replace=False))
    else:
        st_sub = st

    acg = compute_acg_vectorized(st_sub, ACG_BIN_SIZE_MS, ACG_MAX_LAG_MS, fs)

    feats = {
        'firing_rate_hz':    fr,
        'isi_mean_ms':       float(np.mean(isis)) if len(isis) > 0 else 0.0,
        'isi_std_ms':        float(np.std(isis)) if len(isis) > 0 else 0.0,
        'isi_cv':            float(np.std(isis)/np.mean(isis)) if np.mean(isis) > 0 else 0.0,
        'isi_skew':          float(stats.skew(isis)) if len(isis) > 3 else 0.0,
        'presence_ratio':    presence,
        'isi_violation_rate': isi_vr,
        'burst_index':       float(np.mean(isis < 10)),
        'spike_count':       len(st),
    }
    for i, v in enumerate(acg):
        feats[f'acg_{i:02d}'] = float(v)
    return feats


# ════════════════════════════════════════════════════
#  BLOCK E — Recording Context (one per recording)
# ════════════════════════════════════════════════════

def extract_block_e(recording_extractor, n_units, study_set=''):
    """Broadcast to all units in this recording."""
    fs = recording_extractor.get_sampling_frequency()
    dur = recording_extractor.get_num_samples() / fs
    n_ch = recording_extractor.get_num_channels()

    # Noise on random channel sample (fast)
    chunk = recording_extractor.get_traces(
        start_frame=0, end_frame=min(int(fs * 10), recording_extractor.get_num_samples())
    ).astype(np.float32)
    noise_levels = np.median(np.abs(chunk), axis=0) / 0.6745

    return {
        'rec_noise_mean_uv':   float(np.mean(noise_levels)),
        'rec_noise_std_uv':    float(np.std(noise_levels)),
        'rec_duration_sec':    dur,
        'rec_n_channels':      n_ch,
        'rec_n_units':         n_units,
        'rec_sampling_rate':   fs,
    }


# ════════════════════════════════════════════════════
#  BLOCK F — Pairwise Relational (only when n_units ≥ 2)
# ════════════════════════════════════════════════════

def cosine_amp_normalised(tA, tB):
    """Amplitude-normalised cosine — measures shape, not amplitude."""
    pA = float(np.max(np.abs(tA))) + 1e-8
    pB = float(np.max(np.abs(tB))) + 1e-8
    A  = (tA / pA).ravel().astype(np.float32)
    B  = (tB / pB).ravel().astype(np.float32)
    return float(np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B) + 1e-8))

def extract_block_f(unit_id, templates_dict, channel_locations):
    """templates_dict: {unit_id: (n_ch, n_t) float16}."""
    if len(templates_dict) < 2:
        return {'max_cosine': 0.0, 'mean_cosine_top3': 0.0,
                'n_confusable': 0, 'isolation_score': 1.0,
                'min_neighbor_dist_um': np.nan, 'mean_neighbor_dist_um': np.nan}

    tA      = templates_dict[unit_id].astype(np.float32)
    cos_vals = []
    dists    = []
    peak_A   = int(np.argmax(np.ptp(tA, axis=1)))

    for uid_B, tB_f16 in templates_dict.items():
        if uid_B == unit_id:
            continue
        tB    = tB_f16.astype(np.float32)
        cos   = cosine_amp_normalised(tA, tB)
        cos_vals.append(cos)
        if channel_locations is not None:
            peak_B = int(np.argmax(np.ptp(tB, axis=1)))
            d = float(np.linalg.norm(
                channel_locations[peak_A] - channel_locations[peak_B]))
            dists.append(d)
        del tB

    cos_vals = np.array(cos_vals)
    top3     = float(np.mean(np.sort(cos_vals)[-3:])) if len(cos_vals) >= 3 else float(np.max(cos_vals))
    return {
        'max_cosine':            float(np.max(cos_vals)),
        'mean_cosine_top3':      top3,
        'n_confusable':          int(np.sum(cos_vals > COSINE_THRESHOLD)),
        'isolation_score':       float(1.0 - np.max(cos_vals)),
        'min_neighbor_dist_um':  float(np.min(dists)) if dists else np.nan,
        'mean_neighbor_dist_um': float(np.mean(dists)) if dists else np.nan,
    }

print('✓ Feature functions defined (Blocks A, B, C, E, F)')


## 🌲 4. Phase 0 — SpikeForest Enumeration

In [ ]:
# ── Enumerate available sorting outputs ────────────
SPIKEFOREST_MANIFEST = MANIFEST_DIR / "spikeforest_manifest.json"


def _load_cached_manifest_df():
    if SPIKEFOREST_MANIFEST.exists():
        with open(SPIKEFOREST_MANIFEST) as f:
            return pd.DataFrame(json.load(f))
    return pd.DataFrame()


def _local_bootstrap_kachery_client_keys():
    try:
        import kachery_cloud._client_keys as ck
        ck._get_client_keys_hex(generate_if_missing=True)
        pub, priv = ck._get_client_keys_hex(generate_if_missing=False)
        if pub is not None and priv is not None:
            return True
    except Exception:
        pass

    try:
        import kachery_cloud as kcl
        init_fn = getattr(kcl, "init", None)
        if callable(init_fn):
            init_fn()
        import kachery_cloud._client_keys as ck
        ck._get_client_keys_hex(generate_if_missing=True)
        pub, priv = ck._get_client_keys_hex(generate_if_missing=False)
        return pub is not None and priv is not None
    except Exception:
        return False


def _pick_val_local(obj, candidates):
    if isinstance(obj, dict):
        for name in candidates:
            if name in obj and obj[name] is not None:
                return obj[name]
    for name in candidates:
        if hasattr(obj, name):
            val = getattr(obj, name)
            if callable(val):
                try:
                    val = val()
                except Exception:
                    continue
            if val is not None:
                return val
    for meth in ["to_dict", "dict"]:
        if hasattr(obj, meth):
            fn = getattr(obj, meth)
            if callable(fn):
                try:
                    d = fn()
                    if isinstance(d, dict):
                        for name in candidates:
                            if name in d and d[name] is not None:
                                return d[name]
                except Exception:
                    pass
    return None


def _load_spikeforest_recordings_compat():
    def _load_one(uri=None):
        try:
            return sf.load_spikeforest_recordings(uri) if uri else sf.load_spikeforest_recordings()
        except TypeError:
            return sf.load_spikeforest_recordings(uri=uri)
    recs = list(_load_one(SPIKEFOREST_RECORDINGS_URI))
    for _, rec_uri in (SPIKEFOREST_EXTRA_URI_PAIRS or []):
        if rec_uri:
            try:
                recs.extend(_load_one(rec_uri))
            except Exception as _e:
                log_status(f"Warning: could not load extra recordings URI {rec_uri}: {_e}")
    return recs


def _load_spikeforest_sorting_outputs_compat():
    def _load_one(uri=None):
        try:
            return sf.load_spikeforest_sorting_outputs(uri) if uri else sf.load_spikeforest_sorting_outputs()
        except TypeError:
            return sf.load_spikeforest_sorting_outputs(uri=uri)
    outputs = list(_load_one(SPIKEFOREST_SORTING_OUTPUTS_URI))
    for out_uri, _ in (SPIKEFOREST_EXTRA_URI_PAIRS or []):
        if out_uri:
            try:
                outputs.extend(_load_one(out_uri))
            except Exception as _e:
                log_status(f"Warning: could not load extra sorting outputs URI {out_uri}: {_e}")
    return outputs


def _build_study_set_lookup():
    lookup = {}
    try:
        recs = _load_spikeforest_recordings_compat()
        for R in recs:
            study_name = _pick_val_local(R, ["study_name", "study", "studyName"])
            recording_name = _pick_val_local(R, ["recording_name", "recording", "recordingName", "name"])
            study_set = _pick_val_local(R, ["study_set_name", "study_set", "studySetName"])
            if study_name is None or recording_name is None or study_set is None:
                continue
            lookup[(str(study_name), str(recording_name))] = str(study_set)
    except Exception as exc:
        log_status(f"Study-set lookup warning: {exc}")
    return lookup


def _resolve_sf_output_meta(out, study_lookup):
    study_name = _pick_val_local(out, ["study_name", "study", "studyName"])
    recording_name = _pick_val_local(out, ["recording_name", "recording", "recordingName", "name"])
    sorter_name = _pick_val_local(out, ["sorter_name", "sorter", "sorterName"])
    study_set = _pick_val_local(out, ["study_set_name", "study_set", "studySetName"])
    if study_set is None and study_name is not None and recording_name is not None:
        study_set = study_lookup.get((str(study_name), str(recording_name)))

    missing = [k for k, v in {
        "study_set": study_set,
        "study_name": study_name,
        "recording_name": recording_name,
        "sorter_name": sorter_name,
    }.items() if v is None]
    if missing:
        raise RuntimeError(f"Missing metadata {missing} for SFSortingOutput")

    return {
        "study_set": str(study_set),
        "study_name": str(study_name),
        "recording_name": str(recording_name),
        "sorter_name": str(sorter_name),
    }


def _enumerate_sf_manifest_and_lookup():
    all_outputs_local = _load_spikeforest_sorting_outputs_compat()
    study_lookup = _build_study_set_lookup()
    selected_manifest = []
    selected_lookup = {}
    available_study_sets = {}
    skipped_missing = 0

    for out in all_outputs_local:
        try:
            meta = _resolve_sf_output_meta(out, study_lookup)
        except Exception:
            skipped_missing += 1
            continue

        available_study_sets[meta["study_set"]] = available_study_sets.get(meta["study_set"], 0) + 1

        if meta["study_set"] not in SPIKEFOREST_STUDY_SETS:
            continue
        # Leakage guard: skip any training recording that appears in the test set
        if meta["recording_name"] in (TRAIN_EXCLUDE_RECORDING_NAMES or set()):
            log_status(
                f"Leakage guard: skipping training item "
                f"{meta['study_set']}/{meta['recording_name']} (matches test set)"
            )
            continue
        selected_manifest.append(meta)
        key = sf_output_key(meta["study_set"], meta["study_name"], meta["recording_name"], meta["sorter_name"])
        selected_lookup[key] = out

    return selected_manifest, selected_lookup, available_study_sets, skipped_missing


bootstrap_keys_fn = globals().get("bootstrap_kachery_client_keys", _local_bootstrap_kachery_client_keys)
selected = []
selected_sf_objects_by_key = {}
phase0_mode = "none"
available_study_sets = {}
skipped_missing = 0

if TRAIN_PARQUET.exists() and REUSE_EXISTING_TRAIN_FEATURES and not FORCE_REBUILD_TRAIN_FEATURES:
    print("Using existing training parquet; SpikeForest enumeration is optional and skipped.")
    df_manifest = _load_cached_manifest_df()
    if not df_manifest.empty:
        selected = df_manifest.to_dict(orient="records")
    phase0_mode = "skipped_reuse_train"
elif not SPIKEFOREST_AVAILABLE:
    df_manifest = _load_cached_manifest_df()
    if not df_manifest.empty:
        print("⚠ spikeforest not importable — using cached manifest")
        selected = df_manifest.to_dict(orient="records")
        phase0_mode = "cached_manifest"
    else:
        print("⚠ spikeforest not available and no cached manifest found")
        phase0_mode = "unavailable"
else:
    print("Enumerating SpikeForest sorting outputs...")
    try:
        selected, selected_sf_objects_by_key, available_study_sets, skipped_missing = _enumerate_sf_manifest_and_lookup()
        write_json(SPIKEFOREST_MANIFEST, selected)
        df_manifest = pd.DataFrame(selected)
        phase0_mode = "enumerated"
    except Exception as exc:
        msg = str(exc)
        key_error = "client keys" in msg.lower() or "client key" in msg.lower()
        if key_error:
            print("⚠ SpikeForest requires kachery client keys. Attempting automatic key bootstrap...")
            boot_ok = bootstrap_keys_fn()
            if boot_ok:
                try:
                    selected, selected_sf_objects_by_key, available_study_sets, skipped_missing = _enumerate_sf_manifest_and_lookup()
                    write_json(SPIKEFOREST_MANIFEST, selected)
                    df_manifest = pd.DataFrame(selected)
                    phase0_mode = "enumerated_after_key_bootstrap"
                except Exception as exc2:
                    print(f"⚠ Enumeration still failed after key bootstrap: {exc2}")
                    df_manifest = _load_cached_manifest_df()
                    if not df_manifest.empty:
                        selected = df_manifest.to_dict(orient="records")
                    phase0_mode = "cached_manifest_after_failed_bootstrap"
            else:
                print("⚠ Could not bootstrap kachery client keys automatically.")
                df_manifest = _load_cached_manifest_df()
                if not df_manifest.empty:
                    selected = df_manifest.to_dict(orient="records")
                phase0_mode = "cached_manifest_no_keys"
        else:
            print(f"⚠ SpikeForest enumeration failed: {exc}")
            df_manifest = _load_cached_manifest_df()
            if not df_manifest.empty:
                selected = df_manifest.to_dict(orient="records")
            phase0_mode = "cached_manifest_after_error"

if isinstance(selected, list) and len(selected) > 0:
    df_manifest = pd.DataFrame(selected)
else:
    df_manifest = pd.DataFrame()

if skipped_missing > 0:
    log_status(f"Phase 0 skipped {skipped_missing} outputs with unresolved metadata")

if available_study_sets:
    top_sets = sorted(available_study_sets.items(), key=lambda x: x[1], reverse=True)[:20]
    rprint(f"[cyan]Available study sets (top):[/cyan] {top_sets}")

if not df_manifest.empty:
    table = Table(title="SpikeForest Manifest", style="cyan")
    table.add_column("Study Set")
    table.add_column("Sorter")
    table.add_column("N recordings", justify="right")
    for (ss, sorter), grp in df_manifest.groupby(["study_set", "sorter_name"]):
        table.add_row(ss, sorter, str(len(grp)))
    console.print(table)
    log_status(
        f"Phase 0 ({phase0_mode}) complete. {len(df_manifest)} sorter outputs across "
        f"{df_manifest['recording_name'].nunique()} recordings"
    )
else:
    requested = sorted(list(SPIKEFOREST_STUDY_SETS))
    available = sorted(list(available_study_sets.keys())) if available_study_sets else []
    log_status(
        f"Phase 0 ({phase0_mode}) produced no rows after filtering. "
        f"Requested study sets: {requested}. Available study sets (sample): {available[:20]}"
    )



In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PHASE 1 — Feature Extraction from SpikeForest (Training Data)
# ══════════════════════════════════════════════════════════════════

# ── Kachery download timeout patch ─────────────────────────────────────
# kachery-cloud's load_file.py calls requests.get(url, stream=True) with
# NO read timeout, so a stalled TCP connection hangs forever.
# We monkeypatch requests.get to add timeout=(connect_s, read_s) so that
# any chunk-level stall raises requests.exceptions.ReadTimeout, which the
# per-item except clause below catches and logs to failed_keys.
#   connect_s = 60  : raise if server doesn't accept the connection in 60s
#   read_s    = 300 : raise if no data chunk arrives for 5 minutes
# This does NOT limit total download time — only per-chunk idle time.
_KACHERY_CONNECT_TIMEOUT = 60
_KACHERY_READ_TIMEOUT    = 300
_KACHERY_MAX_RETRIES     = 3   # retry transient stalls before giving up

import requests as _requests_mod
_orig_requests_get = _requests_mod.get

def _patched_requests_get(url, **kwargs):
    if "timeout" not in kwargs:
        kwargs["timeout"] = (_KACHERY_CONNECT_TIMEOUT, _KACHERY_READ_TIMEOUT)
    return _orig_requests_get(url, **kwargs)

_requests_mod.get = _patched_requests_get
# Also patch inside already-imported kachery_cloud submodules
for _mod_name in list(__import__("sys").modules):
    if "kachery" in _mod_name:
        _mod = __import__("sys").modules[_mod_name]
        if hasattr(_mod, "requests") and hasattr(_mod.requests, "get"):
            _mod.requests.get = _patched_requests_get
log_status("kachery requests.get patched with download timeout "
           f"(connect={_KACHERY_CONNECT_TIMEOUT}s, read={_KACHERY_READ_TIMEOUT}s, "
           f"max_retries={_KACHERY_MAX_RETRIES})")

# Storage behavior:
# - Each SpikeForest recording/sorter output is processed one at a time.
# - Its extracted features are saved immediately to TRAIN_PARTS_DIR on Drive.
# - On rerun, completed parts are skipped automatically unless
#   FORCE_REBUILD_TRAIN_FEATURES=True.

if FORCE_REBUILD_TRAIN_FEATURES:
    reset_path(TRAIN_PARQUET)
    reset_path(TRAIN_FEATURE_STATE_JSON)
    reset_path(TRAIN_PARTS_DIR)
    TRAIN_PARTS_DIR.mkdir(parents=True, exist_ok=True)

if TRAIN_PARQUET.exists() and REUSE_EXISTING_TRAIN_FEATURES and not FORCE_REBUILD_TRAIN_FEATURES:
    df_train = pd.read_parquet(TRAIN_PARQUET)
    log_status(f"Phase 1 skipped. Reusing training features from {TRAIN_PARQUET}")
elif not SPIKEFOREST_AVAILABLE:
    raise RuntimeError(
        "SpikeForest is required to build the training parquet from scratch. "
        "Re-run the install cell, then resume."
    )
elif "selected" not in globals() or len(selected) == 0:
    raise RuntimeError(
        "Run Phase 0 first. If Phase 0 produced zero rows, check SPIKEFOREST_STUDY_SETS and optional SpikeForest URI overrides in config, or enable REUSE_EXISTING_TRAIN_FEATURES with an existing TRAIN_PARQUET."
    )
else:
    TRAIN_PARTS_DIR.mkdir(parents=True, exist_ok=True)
    state = read_json(
        TRAIN_FEATURE_STATE_JSON,
        default={
            "started_at": datetime.now().isoformat(),
            "completed_keys": [],
            "failed_keys": {},
            "n_parts": 0,
            "complete": False,
        },
    )
    completed = set(state.get("completed_keys", []))
    start_time = time.time()

    # Write startup heartbeat immediately so stalls are visible on Drive.
    state["heartbeat_at"] = datetime.now().isoformat()
    state["last_status"] = "phase1_started"
    write_json(TRAIN_FEATURE_STATE_JSON, state)

    selected_to_process = list(selected)
    if PHASE1_MAX_ITEMS is not None:
        selected_to_process = selected_to_process[: int(PHASE1_MAX_ITEMS)]
        log_status(f"Phase 1 debug limit active: processing first {len(selected_to_process)} items")

    if "selected_sf_objects_by_key" not in globals() or not isinstance(selected_sf_objects_by_key, dict):
        selected_sf_objects_by_key = {}

    def _pick_val_local(obj, candidates):
        if isinstance(obj, dict):
            for name in candidates:
                if name in obj and obj[name] is not None:
                    return obj[name]
        for name in candidates:
            if hasattr(obj, name):
                val = getattr(obj, name)
                if callable(val):
                    try:
                        val = val()
                    except Exception:
                        continue
                if val is not None:
                    return val
        return None

    def _build_study_set_lookup():
        lookup = {}
        try:
            recs = sf.load_spikeforest_recordings()
            for R in recs:
                study_name = _pick_val_local(R, ["study_name", "study", "studyName"])
                recording_name = _pick_val_local(R, ["recording_name", "recording", "recordingName", "name"])
                study_set = _pick_val_local(R, ["study_set_name", "study_set", "studySetName"])
                if study_name is None or recording_name is None or study_set is None:
                    continue
                lookup[(str(study_name), str(recording_name))] = str(study_set)
        except Exception as exc:
            log_status(f"Phase 1 study-set lookup warning: {exc}")
        return lookup

    study_lookup = _build_study_set_lookup()

    def _resolve_sf_meta(item):
        if isinstance(item, dict):
            return {
                "study_set": str(item.get("study_set", "")),
                "study_name": str(item.get("study_name", "")),
                "recording_name": str(item.get("recording_name", "")),
                "sorter_name": str(item.get("sorter_name", "")),
            }

        study_name = _pick_val_local(item, ["study_name", "study", "studyName"])
        recording_name = _pick_val_local(item, ["recording_name", "recording", "recordingName", "name"])
        sorter_name = _pick_val_local(item, ["sorter_name", "sorter", "sorterName"])
        study_set = _pick_val_local(item, ["study_set_name", "study_set", "studySetName"])
        if study_set is None and study_name is not None and recording_name is not None:
            study_set = study_lookup.get((str(study_name), str(recording_name)))

        missing = [k for k, v in {
            "study_set": study_set,
            "study_name": study_name,
            "recording_name": recording_name,
            "sorter_name": sorter_name,
        }.items() if v is None]
        if missing:
            raise RuntimeError(f"Missing metadata {missing} for selected SpikeForest item")

        return {
            "study_set": str(study_set),
            "study_name": str(study_name),
            "recording_name": str(recording_name),
            "sorter_name": str(sorter_name),
        }

    def _load_sf_recording_compat(study_name, recording_name):
        # API compatibility across spikeforest versions
        try:
            return sf.load_spikeforest_recording(study_name=study_name, recording_name=recording_name)
        except TypeError:
            return sf.load_spikeforest_recording(study_name=study_name, recording_name=recording_name, uri=None)

    if len(selected_sf_objects_by_key) == 0 and SPIKEFOREST_AVAILABLE:
        try:
            outputs_for_lookup = sf.load_spikeforest_sorting_outputs()
            for out in outputs_for_lookup:
                try:
                    meta_lookup = _resolve_sf_meta(out)
                    key_lookup = sf_output_key(
                        meta_lookup["study_set"],
                        meta_lookup["study_name"],
                        meta_lookup["recording_name"],
                        meta_lookup["sorter_name"],
                    )
                    selected_sf_objects_by_key[key_lookup] = out
                except Exception:
                    continue
        except Exception as exc:
            log_status(f"Phase 1 lookup warmup warning: {exc}")

    with Progress(
        SpinnerColumn(),
        TextColumn("{task.description}"),
        BarColumn(),
        TextColumn("{task.completed}/{task.total}"),
        TimeElapsedColumn(),
        TimeRemainingColumn(),
    ) as progress:
        task = progress.add_task("[cyan]Extracting features...", total=len(selected_to_process))

        for item in selected_to_process:
            meta = _resolve_sf_meta(item)
            key = sf_output_key(meta["study_set"], meta["study_name"], meta["recording_name"], meta["sorter_name"])
            part_path = TRAIN_PARTS_DIR / f"{safe_stem(key)}.parquet"
            progress.update(task, description=f"[cyan]{key[:70]}")

            if RESUME_FROM_CHECKPOINTS and key in completed and part_path.exists():
                progress.advance(task)
                continue

            state["heartbeat_at"] = datetime.now().isoformat()
            state["current_key"] = key
            state["last_status"] = "processing"
            write_json(TRAIN_FEATURE_STATE_JSON, state)

            last_exc = None
            for _attempt in range(1, _KACHERY_MAX_RETRIES + 1):
              try:
                sf_obj = None if isinstance(item, dict) else item
                if sf_obj is None:
                    sf_obj = selected_sf_objects_by_key.get(key)
                if sf_obj is None:
                    raise RuntimeError(
                        "SpikeForest sorting output object unavailable for this manifest row. "
                        "This usually means auth/enumeration failed in Phase 0."
                    )

                R = _load_sf_recording_compat(meta["study_name"], meta["recording_name"])
                recording = R.get_recording_extractor()
                sorting_gt = R.get_sorting_true_extractor()
                sorting_out = load_sf_sorting_extractor(sf_obj)

                rows = compare_and_extract_rows(
                    recording,
                    sorting_gt,
                    sorting_out,
                    metadata={
                        "study_set": meta["study_set"],
                        "study_name": meta["study_name"],
                        "recording_name": meta["recording_name"],
                        "sorter_name": meta["sorter_name"],
                    },
                )
                pd.DataFrame(rows).to_parquet(part_path, index=False)
                completed.add(key)
                state["completed_keys"] = sorted(completed)
                state["failed_keys"].pop(key, None)
                state["n_parts"] = len(list(TRAIN_PARTS_DIR.glob("*.parquet")))
                state["heartbeat_at"] = datetime.now().isoformat()
                state["last_status"] = "processed_ok"
                write_json(TRAIN_FEATURE_STATE_JSON, state)

                del recording, sorting_gt, sorting_out
                gc.collect()
                check_memory(label=key)
                last_exc = None
                break  # success — exit retry loop

              except Exception as _exc:
                last_exc = _exc
                is_transient = _is_transient_kachery_error(_exc)
                if is_transient and _attempt < _KACHERY_MAX_RETRIES:
                    wait_s = min(30, 3 * _attempt)
                    log_status(
                        f"  attempt {_attempt}/{_KACHERY_MAX_RETRIES} transient error for {key[:60]}: {_exc} | retry in {wait_s}s"
                    )
                    time.sleep(wait_s)
                    gc.collect()
                    continue
                break  # non-transient error or last retry — fall through

            if last_exc is not None:
                state["failed_keys"][key] = str(last_exc)
                state["heartbeat_at"] = datetime.now().isoformat()
                state["last_status"] = "processed_failed"
                write_json(TRAIN_FEATURE_STATE_JSON, state)
                log_status(f"SKIP {key}: {last_exc}")

            progress.advance(task)

    part_paths = []
    for item in selected_to_process:
        try:
            meta = _resolve_sf_meta(item)
        except Exception:
            continue
        key = sf_output_key(meta["study_set"], meta["study_name"], meta["recording_name"], meta["sorter_name"])
        part_path = TRAIN_PARTS_DIR / f"{safe_stem(key)}.parquet"
        if part_path.exists():
            part_paths.append(part_path)

    if not part_paths:
        raise RuntimeError(
            "No training rows were extracted. Check SpikeForest auth/keys and Phase 0 logs, "
            "or reuse an existing TRAIN_PARQUET."
        )

    df_train = pd.concat([pd.read_parquet(pp) for pp in part_paths], ignore_index=True)
    df_train = df_train.drop_duplicates(
        subset=["study_set", "study_name", "recording_name", "sorter_name", "unit_id"]
    ).reset_index(drop=True)
    if "group_key" not in df_train.columns:
        df_train["group_key"] = df_train.apply(
            lambda row: recording_group_key(row["study_set"], row["study_name"], row["recording_name"]),
            axis=1,
        )
    if "row_uid" not in df_train.columns:
        df_train["row_uid"] = df_train.apply(
            lambda row: make_row_uid(row["study_set"], row["study_name"], row["recording_name"], row["sorter_name"], row["unit_id"]),
            axis=1,
        )
    df_train = stable_sort_frame(df_train, ["group_key", "sorter_name", "unit_id", "row_uid"])
    df_train.to_parquet(TRAIN_PARQUET, index=False)
    state["complete"] = True
    state["finished_at"] = datetime.now().isoformat()
    state["n_rows"] = int(len(df_train))
    state["train_parquet"] = str(TRAIN_PARQUET)
    write_json(TRAIN_FEATURE_STATE_JSON, state)

    elapsed = time.time() - start_time
    log_status(
        f"Phase 1 complete. {len(df_train)} rows | "
        f"{df_train['group_key'].nunique()} groups | elapsed: {elapsed / 60:.1f} min"
    )



## 🔍 5. Data Audit

In [ ]:
df_train = pd.read_parquet(TRAIN_PARQUET)
if "group_key" not in df_train.columns:
    df_train["group_key"] = df_train.apply(
        lambda row: recording_group_key(row["study_set"], row["study_name"], row["recording_name"]),
        axis=1,
    )
if "row_uid" not in df_train.columns:
    df_train["row_uid"] = df_train.apply(
        lambda row: make_row_uid(row["study_set"], row["study_name"], row["recording_name"], row["sorter_name"], row["unit_id"]),
        axis=1,
    )
df_train = stable_sort_frame(df_train, ["group_key", "sorter_name", "unit_id", "row_uid"])

required_cols = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "fmiss", "fpos", "accuracy"]
missing_required = [c for c in required_cols if c not in df_train.columns]
if missing_required:
    raise RuntimeError(f"Training parquet is missing required columns: {missing_required}")

meta_cols = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "group_key", "row_uid"]
target_cols = ["fmiss", "fpos", "accuracy"]
feature_cols = [c for c in df_train.columns if c not in meta_cols + target_cols]

duplicate_rows = int(df_train.duplicated(subset=["study_set", "study_name", "recording_name", "sorter_name", "unit_id"]).sum())
row_uid_duplicates = int(df_train["row_uid"].duplicated().sum())
all_null_features = sorted([c for c in feature_cols if df_train[c].isna().all()])
constant_features = sorted([c for c in feature_cols if df_train[c].nunique(dropna=True) <= 1])
label_missing = {c: int(df_train[c].isna().sum()) for c in target_cols}

rprint("[bold cyan]Dataset summary[/bold cyan]")
rprint(f"  Total rows:           {len(df_train):,}")
rprint(f"  Unique groups:        {df_train['group_key'].nunique():,}")
rprint(f"  Unique recordings:    {df_train['recording_name'].nunique():,}")
rprint(f"  Unique sorters:       {df_train['sorter_name'].nunique():,}")
rprint(f"  Feature columns:      {len(feature_cols):,}")
rprint(f"  Duplicate unit rows:  {duplicate_rows:,}")
rprint(f"  Duplicate row_uid:    {row_uid_duplicates:,}")
rprint(f"  NaN fraction:         {df_train[feature_cols].isna().mean().mean():.3f}")
rprint(f"  Missing labels:       {label_missing}")
rprint(f"  All-null features:    {len(all_null_features)}")
rprint(f"  Constant features:    {len(constant_features)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ["accuracy", "fmiss", "fpos"], ["steelblue", "tomato", "seagreen"]):
    vals = df_train[col].dropna()
    ax.hist(vals, bins=50, color=color, alpha=0.85, edgecolor="none")
    ax.axvline(0.3, ls="--", c="k", lw=1.2, label="0.3")
    ax.axvline(0.7, ls="--", c="orange", lw=1.2, label="0.7")
    midrange = float((vals.between(0.3, 0.7).sum() / max(len(vals), 1)) * 100)
    ax.set_title(f"{col} | midrange {midrange:.1f}%")
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
plt.suptitle("Target Variable Distributions", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "target_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

table = Table(title="Rows per sorter × study set", style="magenta")
table.add_column("Sorter")
table.add_column("Study Set")
table.add_column("N rows", justify="right")
table.add_column("N groups", justify="right")
for (sorter, ss), grp in df_train.groupby(["sorter_name", "study_set"]):
    table.add_row(sorter, ss, str(len(grp)), str(grp["group_key"].nunique()))
console.print(table)

write_json(
    DATA_AUDIT_JSON,
    {
        "generated_at": datetime.now().isoformat(),
        "n_rows": int(len(df_train)),
        "n_groups": int(df_train["group_key"].nunique()),
        "n_recordings": int(df_train["recording_name"].nunique()),
        "n_sorters": int(df_train["sorter_name"].nunique()),
        "duplicate_rows": duplicate_rows,
        "duplicate_row_uid": row_uid_duplicates,
        "label_missing": label_missing,
        "all_null_features": all_null_features,
        "constant_features": constant_features,
    },
)
rprint(f"[green]✓ Dataset audit saved to {DATA_AUDIT_JSON}[/green]")


## 🛡️ 5b. Leakage Review

In [ ]:
df_leak = df_train.copy()
if "group_key" not in df_leak.columns:
    df_leak["group_key"] = make_group_ids(df_leak)

raw_name_collisions = (
    df_leak[["study_set", "study_name", "recording_name"]]
    .drop_duplicates()
    .groupby("recording_name")[["study_set", "study_name"]]
    .nunique()
)
raw_name_collisions = raw_name_collisions[
    (raw_name_collisions["study_set"] > 1) | (raw_name_collisions["study_name"] > 1)
]

recording_level_cols = sorted([c for c in df_leak.columns if c.startswith("rec_")])

risk_rows = [
    {
        "risk": "Imputation before CV split",
        "status": "FIXED",
        "details": "Fold-specific medians are now fit on each outer training fold only.",
        "recommended_validation": "Keep this as the default. For external inference, store either fold medians or a final full-training imputer.",
    },
    {
        "risk": "Early stopping on the scored outer fold",
        "status": "FIXED",
        "details": "Early stopping now uses an inner group-aware split inside each outer training fold.",
        "recommended_validation": "If you want the strictest estimate, report outer-fold metrics only and keep inner-fold metrics for diagnostics.",
    },
    {
        "risk": "Grouping only by recording_name",
        "status": "FIXED",
        "details": (
            "Raw recording_name collisions were detected across studies, so the notebook now groups by study_set + study_name + recording_name."
            if not raw_name_collisions.empty
            else "No raw-name collisions were found in the current parquet, but composite grouping is still safer and now enforced."
        ),
        "recommended_validation": "Use composite recording groups as the minimum standard. For a harsher check, run leave-one-study-set-out validation.",
    },
    {
        "risk": "Recording-level features repeated for all units",
        "status": "MONITOR",
        "details": f"{len(recording_level_cols)} rec_* features are repeated within a group. This is valid only because the split now holds out entire composite recordings.",
        "recommended_validation": "Keep grouped CV. As a sensitivity test, train once with rec_* features dropped and compare metrics.",
    },
    {
        "risk": "Synthetic-to-sim_hybrid domain shift",
        "status": "NOT LEAKAGE",
        "details": "This is not leakage, but it is a real external-validity risk because the training data are synthetic SpikeForest studies and the test example is sim_hybrid.",
        "recommended_validation": "Run robustness checks such as leave-one-study-set-out and report external-test performance separately from cross-validation.",
    },
    {
        "risk": "Sorter-specific overfitting",
        "status": "NOT LEAKAGE",
        "details": "Even without explicit sorter_name features, the model may learn sorter-specific signatures through morphology and train quality patterns.",
        "recommended_validation": "Run leave-one-sorter-out CV as a secondary analysis if sorter generalization matters for the thesis claim.",
    },
]

display(pd.DataFrame(risk_rows))
if not raw_name_collisions.empty:
    rprint("[yellow]Raw recording_name collisions detected across study contexts:[/yellow]")
    display(raw_name_collisions.reset_index().head(20))

validation_options = [
    {
        "name": "Default recommended",
        "protocol": "Outer GroupKFold on composite recording groups, inner GroupShuffleSplit for early stopping",
        "best_for": "Balanced estimate of unit-level performance with leakage control",
    },
    {
        "name": "Leave-one-sorter-out",
        "protocol": "Hold out all rows from one sorter_name at a time",
        "best_for": "Testing whether the model generalises to unseen sorter families",
    },
    {
        "name": "Leave-one-study-set-out",
        "protocol": "Hold out one study_set at a time",
        "best_for": "Testing whether the model generalises across simulation generators / acquisition conditions",
    },
]
display(pd.DataFrame(validation_options))

write_json(
    LEAKAGE_REPORT_JSON,
    {
        "generated_at": datetime.now().isoformat(),
        "risks": risk_rows,
        "raw_name_collisions": raw_name_collisions.reset_index().to_dict(orient="records"),
        "recording_level_features": recording_level_cols,
        "validation_options": validation_options,
    },
)
rprint(f"[yellow]Leakage review saved to {LEAKAGE_REPORT_JSON}[/yellow]")


## 🏋️ 6. Model Training — Leakage-Controlled Baseline

In [ ]:
META_COLS   = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "group_key", "row_uid"]
TARGET_COLS = ["fmiss", "fpos", "accuracy"]
FEAT_COLS   = [c for c in df_train.columns if c not in META_COLS + TARGET_COLS and not c.startswith("rec_n_units")]
rprint(f"[cyan]Feature columns:[/cyan] {len(FEAT_COLS)}")
if len(FEAT_COLS) == 0:
    raise RuntimeError("No feature columns were found in the training parquet.")

if FORCE_RETRAIN_MODELS:
    for target_name in ["fmiss", "fpos"]:
        reset_path(RESULTS_DIR / f"cv_{target_name}.csv")
        for path in MODEL_DIR.glob(f"model_{target_name}_fold*.json"):
            reset_path(path)
        for path in OOF_DIR.glob(f"{target_name}_fold*.npz"):
            reset_path(path)
    for path in PREPROCESS_DIR.glob("fold*.json"):
        reset_path(path)
    reset_path(CV_STATE_JSON)
    reset_path(OOF_PREDICTIONS_PARQUET)

df_model = df_train[META_COLS + FEAT_COLS + TARGET_COLS].copy()
df_model[FEAT_COLS] = df_model[FEAT_COLS].apply(pd.to_numeric, errors="coerce")
df_model = stable_sort_frame(df_model, ["group_key", "sorter_name", "unit_id", "row_uid"])

full_train_medians = df_model[FEAT_COLS].median(numeric_only=True)
write_json(FULL_TRAIN_MEDIANS_JSON, {k: float(v) for k, v in full_train_medians.items()})
write_json(FEATURE_COLUMNS_JSON, FEAT_COLS)

groups = df_model["group_key"].astype(str).values
y_targets = {
    "fmiss": df_model["fmiss"].values.astype(np.float32),
    "fpos": df_model["fpos"].values.astype(np.float32),
}

unique_groups = np.unique(groups)
n_splits = min(CV_FOLDS, len(unique_groups))
if n_splits < 2:
    raise RuntimeError("GroupKFold requires at least two unique composite recordings.")
if n_splits != CV_FOLDS:
    rprint(f"[yellow]⚠ Reducing CV folds from {CV_FOLDS} to {n_splits} because only {len(unique_groups)} groups are available.[/yellow]")

gkf = GroupKFold(n_splits=n_splits)
models = {"fmiss": [], "fpos": []}
all_val_pred = {
    "fmiss": np.full(len(df_model), np.nan, dtype=np.float32),
    "fpos": np.full(len(df_model), np.nan, dtype=np.float32),
}
fold_assignments = np.full(len(df_model), -1, dtype=np.int16)

for fold, (tr_idx, val_idx) in enumerate(gkf.split(df_model[FEAT_COLS], y_targets["fmiss"], groups)):
    fold_assignments[val_idx] = fold
    df_tr = df_model.iloc[tr_idx].copy()
    df_val = df_model.iloc[val_idx].copy()
    leakage_audit(df_tr, df_val, fold)

    fold_medians = df_tr[FEAT_COLS].median(numeric_only=True)
    write_json(
        PREPROCESS_DIR / f"fold{fold}.json",
        {
            "fold": fold,
            "feature_medians": {k: float(v) for k, v in fold_medians.items()},
            "group_count_train": int(df_tr["group_key"].nunique()),
            "group_count_val": int(df_val["group_key"].nunique()),
        },
    )

    X_tr = df_tr[FEAT_COLS].fillna(fold_medians).values.astype(np.float32)
    X_val = df_val[FEAT_COLS].fillna(fold_medians).values.astype(np.float32)
    inner_split = make_inner_group_split(df_tr["group_key"].values, random_state=RAND_SEED + fold)

    for target_name in ["fmiss", "fpos"]:
        y_tr = y_targets[target_name][tr_idx]
        y_val = y_targets[target_name][val_idx]
        model_path = MODEL_DIR / f"model_{target_name}_fold{fold}.json"
        cached_pred = load_oof_prediction(target_name, fold)

        if RESUME_FROM_CHECKPOINTS and model_path.exists() and cached_pred is not None and not FORCE_RETRAIN_MODELS:
            model = xgb.XGBRegressor()
            model.load_model(str(model_path))
            y_hat = cached_pred["preds"]
            all_val_pred[target_name][cached_pred["val_idx"]] = y_hat
            models[target_name].append(model)
            row = {
                "fold": fold,
                "mae": float(mean_absolute_error(y_val[~np.isnan(y_val)], y_hat)),
                "r2": float(r2_score(y_val[~np.isnan(y_val)], y_hat)) if (~np.isnan(y_val)).sum() > 1 else np.nan,
                "spearman": float(spearmanr(y_val[~np.isnan(y_val)], y_hat).statistic) if (~np.isnan(y_val)).sum() > 1 else np.nan,
            }
            upsert_metric_csv(target_name, row)
            rprint(f"[cyan]Resumed fold {fold} [{target_name}] from checkpoint[/cyan]")
            continue

        mask_tr = ~np.isnan(y_tr)
        mask_val = ~np.isnan(y_val)
        if mask_tr.sum() < 10 or mask_val.sum() < 5:
            log_status(
                f"SKIP fold {fold} [{target_name}] — insufficient labeled rows "
                f"(train={mask_tr.sum()}, val={mask_val.sum()})"
            )
            continue

        fit_kwargs = {}
        if inner_split is not None:
            inner_tr_idx, inner_es_idx = inner_split
            inner_mask_tr = mask_tr[inner_tr_idx]
            inner_mask_es = mask_tr[inner_es_idx]
            if inner_mask_tr.sum() >= 10 and inner_mask_es.sum() >= 5:
                fit_kwargs["eval_set"] = [
                    (X_tr[inner_es_idx][inner_mask_es], y_tr[inner_es_idx][inner_mask_es])
                ]

        model_params = dict(XGB_PARAMS)
        if "eval_set" in fit_kwargs:
            model_params["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
            X_fit = X_tr[inner_tr_idx][inner_mask_tr]
            y_fit = y_tr[inner_tr_idx][inner_mask_tr]
        else:
            X_fit = X_tr[mask_tr]
            y_fit = y_tr[mask_tr]

        model = xgb.XGBRegressor(**model_params)
        model.fit(X_fit, y_fit, verbose=False, **fit_kwargs)
        y_hat = np.clip(model.predict(X_val[mask_val]), 0, 1)

        all_val_pred[target_name][val_idx[mask_val]] = y_hat
        save_oof_prediction(target_name, fold, val_idx[mask_val], y_hat)
        model.save_model(str(model_path))
        models[target_name].append(model)

        row = {
            "fold": fold,
            "mae": float(mean_absolute_error(y_val[mask_val], y_hat)),
            "r2": float(r2_score(y_val[mask_val], y_hat)) if mask_val.sum() > 1 else np.nan,
            "spearman": float(spearmanr(y_val[mask_val], y_hat).statistic) if mask_val.sum() > 1 else np.nan,
        }
        upsert_metric_csv(target_name, row)
        rprint(f"  Fold {fold} [{target_name}] MAE={row['mae']:.3f} R²={row['r2']:.3f} ρ={row['spearman']:.3f}")

for target_name in ["fmiss", "fpos"]:
    df_res = pd.read_csv(RESULTS_DIR / f"cv_{target_name}.csv")
    console.print(f"\n[bold]{target_name} CV summary[/bold]")
    for metric in ["mae", "r2", "spearman"]:
        vals = df_res[metric].values.astype(float)
        rprint(f"  {metric}: {np.nanmean(vals):.3f} ± {np.nanstd(vals):.3f}")

oof_export = df_model[META_COLS + TARGET_COLS].copy()
oof_export["pred_fmiss"] = all_val_pred["fmiss"]
oof_export["pred_fpos"] = all_val_pred["fpos"]
oof_export.to_parquet(OOF_PREDICTIONS_PARQUET, index=False)
fold_export = df_model[META_COLS + TARGET_COLS].copy()
fold_export["fold"] = fold_assignments
fold_export.to_parquet(FOLD_ASSIGNMENTS_PARQUET, index=False)
write_model_manifest()

write_json(
    CV_STATE_JSON,
    {
        "generated_at": datetime.now().isoformat(),
        "n_splits": int(n_splits),
        "feature_count": int(len(FEAT_COLS)),
        "targets": ["fmiss", "fpos"],
        "oof_predictions": str(OOF_PREDICTIONS_PARQUET),
        "fold_assignments": str(FOLD_ASSIGNMENTS_PARQUET),
    },
)
log_status("Phase 3 complete. Models trained and saved.")


## 📊 7. Training Diagnostics & SHAP

In [ ]:
if "models" not in globals():
    models = {"fmiss": load_saved_models("fmiss"), "fpos": load_saved_models("fpos")}
if "FEAT_COLS" not in globals() or not FEAT_COLS:
    FEAT_COLS = read_json(FEATURE_COLUMNS_JSON, default=[])
if not FEAT_COLS:
    raise RuntimeError("Feature manifest missing. Run the training cell first.")
if "META_COLS" not in globals():
    META_COLS = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "group_key", "row_uid"]
if "TARGET_COLS" not in globals():
    TARGET_COLS = ["fmiss", "fpos", "accuracy"]
if "df_model" not in globals():
    df_model = df_train[META_COLS + FEAT_COLS + TARGET_COLS].copy()
    df_model[FEAT_COLS] = df_model[FEAT_COLS].apply(pd.to_numeric, errors="coerce")
if "all_val_pred" not in globals():
    all_val_pred = {"fmiss": np.full(len(df_model), np.nan, dtype=np.float32), "fpos": np.full(len(df_model), np.nan, dtype=np.float32)}
    cv_state = read_json(CV_STATE_JSON, default={})
    n_splits_saved = int(cv_state.get("n_splits", CV_FOLDS))
    for target_name in ["fmiss", "fpos"]:
        for fold in range(n_splits_saved):
            cached = load_oof_prediction(target_name, fold)
            if cached is not None:
                all_val_pred[target_name][cached["val_idx"]] = cached["preds"]

y_fmiss = df_model["fmiss"].values.astype(np.float32)
y_fpos = df_model["fpos"].values.astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, target_name, y_all in [
    (axes[0], "fmiss", y_fmiss),
    (axes[1], "fpos", y_fpos),
]:
    y_pred = all_val_pred[target_name]
    mask = ~np.isnan(y_pred) & ~np.isnan(y_all)
    yt, yp = y_all[mask], y_pred[mask]
    rho = spearmanr(yt, yp).statistic if len(yt) > 1 else np.nan
    mae = mean_absolute_error(yt, yp) if len(yt) > 0 else np.nan
    ax.hexbin(yt, yp, gridsize=40, cmap="YlOrRd", mincnt=1)
    ax.plot([0, 1], [0, 1], "k--", lw=1.2)
    ax.set(
        xlabel=f"True {target_name}",
        ylabel=f"Predicted {target_name}",
        title=f"{target_name} | ρ={rho:.3f}, MAE={mae:.3f}",
        xlim=(-0.05, 1.05),
        ylim=(-0.05, 1.05),
    )
plt.suptitle("Out-of-Fold Predictions vs Ground Truth", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "scatter_pred_vs_true.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target_name, y_all in [
    (axes[0], "fmiss", y_fmiss),
    (axes[1], "fpos", y_fpos),
]:
    y_pred = all_val_pred[target_name]
    mask = ~np.isnan(y_pred) & ~np.isnan(y_all)
    bins = np.linspace(0, 1, 11)
    bx, by = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (y_pred[mask] >= lo) & (y_pred[mask] < hi)
        if in_bin.sum() >= 5:
            bx.append((lo + hi) / 2)
            by.append(float(np.mean(y_all[mask][in_bin])))
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfect")
    ax.plot(bx, by, "o-", label="Calibration curve")
    ax.set(xlabel="Predicted", ylabel="Mean true", title=f"Calibration — {target_name}")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "calibration.png", dpi=150)
plt.show()

if SHAP_AVAILABLE:
    rprint("[cyan]Computing SHAP values (may take 1-2 min)...[/cyan]")
    for target_name in ["fmiss", "fpos"]:
        if not models.get(target_name):
            continue
        model_0 = models[target_name][0]
        medians_0 = load_fold_medians(0)
        X_full = df_model[FEAT_COLS].fillna(medians_0).values.astype(np.float32)
        idx_sub = make_rng("shap", target_name).choice(len(X_full), min(2000, len(X_full)), replace=False)
        explainer = shap.TreeExplainer(model_0)
        shap_values = explainer.shap_values(X_full[idx_sub])

        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_values, X_full[idx_sub], feature_names=FEAT_COLS, max_display=20, show=False)
        plt.title(f"SHAP Feature Importance — {target_name}", fontweight="bold")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"shap_{target_name}.png", dpi=150, bbox_inches="tight")
        plt.show()

        mean_abs = np.abs(shap_values).mean(axis=0)
        pd.DataFrame({"feature": FEAT_COLS, "shap_importance": mean_abs})            .sort_values("shap_importance", ascending=False)            .to_csv(RESULTS_DIR / f"feature_importance_{target_name}.csv", index=False)
        rprint(f"[green]✓ SHAP saved for {target_name}[/green]")
else:
    rprint(f"[yellow]⚠ Skipping SHAP diagnostics:[/yellow] {SHAP_IMPORT_ERROR}")


## 🧪 8. Test Set — sim_hybrid (Precomputed Kilosort4 supported)

In [ ]:
# Storage behavior:
# - Each test recording/sorter pair is processed one at a time.
# - Its extracted features are saved immediately to TEST_PARTS_DIR on Drive.
# - On rerun, completed parts are skipped automatically unless
#   FORCE_REBUILD_TEST_FEATURES=True.

if FORCE_REBUILD_TEST_FEATURES:
    reset_path(TEST_PARQUET)
    reset_path(TEST_FEATURE_STATE_JSON)
    reset_path(TEST_PARTS_DIR)
    TEST_PARTS_DIR.mkdir(parents=True, exist_ok=True)

if TEST_PARQUET.exists() and REUSE_EXISTING_TEST_FEATURES and not FORCE_REBUILD_TEST_FEATURES:
    df_test = pd.read_parquet(TEST_PARQUET)
    log_status(f"Phase 4 feature extraction skipped. Reusing {TEST_PARQUET}")
else:
    TEST_PARTS_DIR.mkdir(parents=True, exist_ok=True)
    state = read_json(
        TEST_FEATURE_STATE_JSON,
        default={
            "started_at": datetime.now().isoformat(),
            "completed_pairs": [],
            "failed_pairs": {},
            "complete": False,
        },
    )
    completed_pairs = set(state.get("completed_pairs", []))
    test_manifest = []

    with Progress(
        SpinnerColumn(),
        TextColumn("{task.description}"),
        BarColumn(),
        TextColumn("{task.completed}/{task.total}"),
        TimeElapsedColumn(),
        TimeRemainingColumn(),
    ) as progress:
        total_pairs = max(1, sum(max(1, len(TEST_SORTERS)) for _ in TEST_RECORDING_SPECS))
        task = progress.add_task("[cyan]Processing test recordings...", total=total_pairs)

        for spec in TEST_RECORDING_SPECS:
            rec_name = spec["recording_name"]
            try:
                rec_dir = resolve_existing_path([spec["recording_dir"]])
                gt_source = resolve_existing_path(spec["ground_truth_candidates"])
                recording = read_spikeglx_recording(rec_dir)
                sorting_gt = load_ground_truth_sorting(gt_source, recording.get_sampling_frequency())
            except Exception as exc:
                log_status(f"SKIP test {rec_name}: {exc}")
                for _ in TEST_SORTERS:
                    progress.advance(task)
                continue

            precomputed_sorters = {
                sorter_name: Path(sorter_path)
                for sorter_name, sorter_path in spec.get("precomputed_sorters", {}).items()
                if Path(sorter_path).exists()
            }
            requested_sorters = list(TEST_SORTERS)
            if not RUN_TEST_SORTERS:
                requested_sorters = [s for s in requested_sorters if s in precomputed_sorters]
            if not requested_sorters:
                log_status(
                    f"SKIP test {rec_name}: no precomputed sorter output found and RUN_TEST_SORTERS=False."
                )
                continue

            for sorter_name in requested_sorters:
                pair_key = f"{rec_name}/{sorter_name}"
                part_path = TEST_PARTS_DIR / f"{safe_stem(pair_key)}.parquet"
                progress.update(task, description=f"[cyan]{pair_key}")

                if RESUME_FROM_CHECKPOINTS and pair_key in completed_pairs and part_path.exists():
                    test_manifest.append(
                        {
                            "recording_name": rec_name,
                            "sorter_name": sorter_name,
                            "source_kind": "checkpoint",
                            "source_path": str(part_path),
                            "ground_truth_source": str(gt_source),
                            "recording_dir": str(rec_dir),
                        }
                    )
                    progress.advance(task)
                    continue

                try:
                    if sorter_name in precomputed_sorters:
                        sorter_source = precomputed_sorters[sorter_name]
                        sorting_out = load_sorting_output(sorter_name, sorter_source)
                        source_kind = "precomputed"
                    elif RUN_TEST_SORTERS:
                        sorter_source = SORTER_SCRATCH_DIR / rec_name / sorter_name
                        sorter_source.parent.mkdir(parents=True, exist_ok=True)
                        sorting_out = si.run_sorter(
                            sorter_name,
                            recording,
                            folder=str(sorter_source),
                            remove_existing_folder=False,
                            verbose=True,
                        )
                        source_kind = "computed"
                    else:
                        progress.advance(task)
                        continue

                    rows = compare_and_extract_rows(
                        recording,
                        sorting_gt,
                        sorting_out,
                        metadata={
                            "study_set": "TEST",
                            "study_name": "sim_hybrid",
                            "recording_name": rec_name,
                            "sorter_name": sorter_name,
                        },
                    )
                    pd.DataFrame(rows).to_parquet(part_path, index=False)
                    completed_pairs.add(pair_key)
                    state["completed_pairs"] = sorted(completed_pairs)
                    state["failed_pairs"].pop(pair_key, None)
                    write_json(TEST_FEATURE_STATE_JSON, state)
                    test_manifest.append(
                        {
                            "recording_name": rec_name,
                            "sorter_name": sorter_name,
                            "source_kind": source_kind,
                            "source_path": str(sorter_source),
                            "ground_truth_source": str(gt_source),
                            "recording_dir": str(rec_dir),
                            "n_rows": len(rows),
                        }
                    )
                    del sorting_out
                    gc.collect()
                except Exception as exc:
                    state["failed_pairs"][pair_key] = str(exc)
                    write_json(TEST_FEATURE_STATE_JSON, state)
                    log_status(f"SKIP test {pair_key}: {exc}")

                progress.advance(task)

            del recording, sorting_gt
            gc.collect()
            check_memory(label=f"test:{rec_name}")

    part_paths = []
    for spec in TEST_RECORDING_SPECS:
        rec_name = spec["recording_name"]
        for sorter_name in TEST_SORTERS:
            part_path = TEST_PARTS_DIR / f"{safe_stem(f'{rec_name}/{sorter_name}')}.parquet"
            if part_path.exists():
                part_paths.append(part_path)
    if not part_paths:
        raise RuntimeError(
            "No test rows were extracted. Check the Drive paths for the sim_hybrid recording, "
            "ground truth, and precomputed sorter outputs."
        )

    df_test = pd.concat([pd.read_parquet(p) for p in part_paths], ignore_index=True)
    if "group_key" not in df_test.columns:
        df_test["group_key"] = df_test.apply(
            lambda row: recording_group_key(row["study_set"], row["study_name"], row["recording_name"]),
            axis=1,
        )
    if "row_uid" not in df_test.columns:
        df_test["row_uid"] = df_test.apply(
            lambda row: make_row_uid(row["study_set"], row["study_name"], row["recording_name"], row["sorter_name"], row["unit_id"]),
            axis=1,
        )
    df_test = stable_sort_frame(df_test, ["group_key", "sorter_name", "unit_id", "row_uid"])
    df_test.to_parquet(TEST_PARQUET, index=False)
    write_json(TEST_SOURCE_JSON, test_manifest)
    state["complete"] = True
    state["finished_at"] = datetime.now().isoformat()
    state["n_rows"] = int(len(df_test))
    write_json(TEST_FEATURE_STATE_JSON, state)
    log_status(f"Test features saved: {len(df_test)} rows")

if CLEAN_SCRATCH_AFTER_RUN and SORTER_SCRATCH_DIR.exists():
    shutil.rmtree(SORTER_SCRATCH_DIR, ignore_errors=True)
    SORTER_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)


## 📈 9. Test Set Evaluation

In [ ]:
if "models" not in globals():
    models = {"fmiss": load_saved_models("fmiss"), "fpos": load_saved_models("fpos")}
if "META_COLS" not in globals():
    META_COLS = ["study_set", "study_name", "recording_name", "sorter_name", "unit_id", "group_key", "row_uid"]
if "TARGET_COLS" not in globals():
    TARGET_COLS = ["fmiss", "fpos", "accuracy"]
if "FEAT_COLS" not in globals() or not FEAT_COLS:
    with open(FEATURE_COLUMNS_JSON) as f:
        FEAT_COLS = json.load(f)
if any(len(models[target_name]) == 0 for target_name in ["fmiss", "fpos"]):
    raise RuntimeError("Saved fold models were not found. Run the training cell first.")

test_parquet = TEST_PARQUET
if not test_parquet.exists():
    rprint("[yellow]No test features found — run the test feature extraction cell first[/yellow]")
else:
    df_test = pd.read_parquet(test_parquet)
    if "group_key" not in df_test.columns:
        df_test["group_key"] = df_test.apply(
            lambda row: recording_group_key(row["study_set"], row["study_name"], row["recording_name"]),
            axis=1,
        )
    if "row_uid" not in df_test.columns:
        df_test["row_uid"] = df_test.apply(
            lambda row: make_row_uid(row["study_set"], row["study_name"], row["recording_name"], row["sorter_name"], row["unit_id"]),
            axis=1,
        )
    df_test = stable_sort_frame(df_test, ["group_key", "sorter_name", "unit_id", "row_uid"])

    eval_results = []
    per_sorter_results = []
    pred_export = df_test[META_COLS + TARGET_COLS].copy()

    for target_name in ["fmiss", "fpos"]:
        fold_preds = []
        for fold, model in enumerate(models[target_name]):
            medians = load_fold_medians(fold)
            df_test_fold, missing_test_cols = ensure_feature_matrix(df_test, FEAT_COLS, medians)
            X_test_fold = df_test_fold[FEAT_COLS].values.astype(np.float32)
            fold_preds.append(np.clip(model.predict(X_test_fold), 0, 1))
        preds = np.mean(fold_preds, axis=0)
        pred_export[f"pred_{target_name}"] = preds

        y_true = df_test[target_name].values.astype(np.float32)
        mask = ~np.isnan(y_true)
        if mask.sum() == 0:
            log_status(f"SKIP test metrics [{target_name}] — no labels available")
            continue

        mae = mean_absolute_error(y_true[mask], preds[mask])
        r2 = r2_score(y_true[mask], preds[mask]) if mask.sum() > 1 else np.nan
        rho = spearmanr(y_true[mask], preds[mask]).statistic if mask.sum() > 1 else np.nan
        rprint(f"[bold]Test set [{target_name}][/bold] MAE={mae:.3f}  R²={r2:.3f}  ρ={rho:.3f}")
        eval_results.append({"target": target_name, "MAE": mae, "R2": r2, "Spearman": rho})

        for sorter in sorted(df_test["sorter_name"].dropna().unique()):
            sm = (df_test["sorter_name"] == sorter).values & mask
            if sm.sum() < 5:
                continue
            mae_s = mean_absolute_error(y_true[sm], preds[sm])
            rho_s = spearmanr(y_true[sm], preds[sm]).statistic if sm.sum() > 1 else np.nan
            per_sorter_results.append(
                {
                    "target": target_name,
                    "sorter_name": sorter,
                    "MAE": mae_s,
                    "Spearman": rho_s,
                    "n": int(sm.sum()),
                }
            )
            rprint(f"  [{sorter}] MAE={mae_s:.3f}  ρ={rho_s:.3f}  n={sm.sum()}")

    pred_export.to_parquet(TEST_PREDICTIONS_PARQUET, index=False)
    pd.DataFrame(eval_results).to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
    pd.DataFrame(per_sorter_results).to_csv(RESULTS_DIR / "test_metrics_by_sorter.csv", index=False)
    log_status("Phase 4 complete. Test evaluation done.")


## ✅ 10. Final Report

In [ ]:
console.print("[bold cyan]══════ EXPERIMENT SUMMARY ══════[/bold cyan]")
if "models" not in globals():
    models = {"fmiss": load_saved_models("fmiss"), "fpos": load_saved_models("fpos")}
if "FEAT_COLS" not in globals() or not FEAT_COLS:
    FEAT_COLS = read_json(FEATURE_COLUMNS_JSON, default=[])

if TRAIN_PARQUET.exists():
    df_train_final = pd.read_parquet(TRAIN_PARQUET)
    rprint(f"  Training rows:        {len(df_train_final):,}")
    rprint(f"  Training groups:      {df_train_final['group_key'].nunique():,}" if "group_key" in df_train_final.columns else "  Training groups:      N/A")
    rprint(f"  Feature dimensions:   {len(FEAT_COLS)}")
    rprint(f"  CV folds trained:     {len(models.get('fmiss', []))}")

for t in ["fmiss", "fpos"]:
    cv_path = RESULTS_DIR / f"cv_{t}.csv"
    if cv_path.exists():
        df_cv = pd.read_csv(cv_path)
        rprint(
            f"  CV {t}: MAE={df_cv.mae.mean():.3f}±{df_cv.mae.std():.3f} "
            f"R²={df_cv.r2.mean():.3f}±{df_cv.r2.std():.3f} "
            f"ρ={df_cv.spearman.mean():.3f}±{df_cv.spearman.std():.3f}"
        )

test_file = RESULTS_DIR / "test_metrics.csv"
if test_file.exists():
    df_tm = pd.read_csv(test_file)
    console.print("[bold green]Test Set (sim_hybrid)[/bold green]")
    for _, row in df_tm.iterrows():
        rprint(f"  [{row.target}] MAE={row.MAE:.3f}  R²={row.R2:.3f}  ρ={row.Spearman:.3f}")

manifest = write_artifact_manifest()
write_model_manifest()
prototype_size_gb = directory_size_bytes(PROTOTYPE_ROOT) / 1e9
scratch_size_gb = directory_size_bytes(SCRATCH_ROOT) / 1e9

console.print("\n[bold]Checkpoint summary:[/bold]")
rprint(f"  Prototype root:         {PROTOTYPE_ROOT} ({prototype_size_gb:.3f} GB)")
rprint(f"  Scratch root:           {SCRATCH_ROOT} ({scratch_size_gb:.3f} GB)")
rprint(f"  Train feature parts:    {len(list(TRAIN_PARTS_DIR.glob('*.parquet')))}")
rprint(f"  Test feature parts:     {len(list(TEST_PARTS_DIR.glob('*.parquet')))}")
rprint(f"  Train resume state:     {TRAIN_FEATURE_STATE_JSON}")
rprint(f"  Test resume state:      {TEST_FEATURE_STATE_JSON}")
rprint(f"  CV resume state:        {CV_STATE_JSON}")
rprint(f"  Run manifest:           {RUN_MANIFEST_JSON}")
rprint(f"  Source data manifest:   {SOURCE_DATA_MANIFEST_JSON}")
rprint(f"  Model manifest:         {MODEL_MANIFEST_JSON}")
rprint(f"  Fold assignments:       {FOLD_ASSIGNMENTS_PARQUET}")
rprint(f"  Artifact manifest:      {ARTIFACT_MANIFEST_JSON}")
rprint(f"  Leakage report:         {LEAKAGE_REPORT_JSON}")
rprint(f"  Data audit:             {DATA_AUDIT_JSON}")
rprint("  Raw external datasets are referenced in place from /content/drive/MyDrive/Thesis/Data.")
rprint("  Intermediate feature parts and fold-level CV artifacts are persisted on Drive for crash-safe resume.")

console.print("\n[bold]Output files under prototype:[/bold]")
for entry in manifest:
    rel = Path(entry["path"]).relative_to(PROTOTYPE_ROOT)
    rprint(f"  {rel}  ({entry['size_mb']:.2f} MB)")
